# Práctica · La capa refinada del acueducto con DataFrames de Spark

**IFPN0025 · Big Data e Ingeniería de Datos** · Universidad Ean · Módulo 2 · Sesión 16

**Qué hace este cuaderno.** Construye la capa refinada del lago del acueducto con DataFrames de Spark a partir de la capa cruda en CSV, con el esquema declarado, la marca temporal en microsegundos, partición por día y sobrescritura dinámica. Reconcilia el resultado **sin filtro de calidad** contra las referencias del lago, comprueba el tipo de la marca temporal con un lector independiente y decide, con mediciones propias (una corrida de calentamiento descartada y tres medidas), entre una expresión nativa, una función por fila y una función vectorizada para clasificar la presión.

| Nivel | Nombre | Dónde | Tiempo | Qué produce |
|---|---|---|---|---|
| 1 | Guiado | En clase | 45 min | Contraste con la solución de referencia de la carga con Spark; sesión `local[4]`; esquema inferido, declarado y mal declarado; capa nueva reconciliada; tres formas de clasificar con el protocolo de medición; tabla de mediciones |
| 2 | Aplicado | Autónomo · obligatorio | 1,5 h del presupuesto de 4 h de la tarea | Ampliación del equipo; cuatro funciones completadas en `pipeline/refinada_spark.py`; dos corridas por volumen con el mismo hash; reproceso de 6 días; 12 corridas por tabla; cierre diario frente al RDD; evidencia del nivel |
| 3 | Frontera | Autónomo · opcional | Fuera de las 4 h | Política de evolución del esquema ante una fuente que cambia |
| · | Reto de diseño | Autónomo · opcional | 30 min | Qué cuesta la función propia en un cierre de 100 000 medidores, y cómo contárselo a la gerencia |

**Cómo usarlo.** Ejecuten las celdas en orden. El Nivel 1 está resuelto para seguirlo en clase. En el Nivel 2, el Nivel 3 y el reto encontrarán celdas con `# TODO` y `NotImplementedError` que deben completar (al terminar cada una, borren la línea `raise NotImplementedError`); después de cada ejercicio hay una celda de autoverificación con `assert` que les dice si su resultado cumple el criterio de correctitud. La lista de entrega y el `spark.stop()` finales existen solo en este cuaderno (la guía no los reproduce), y las funciones propias `clasificar_udf_fila` y `clasificar_udf_pandas` que muestra la guía ya vienen escritas en `pipeline/refinada_spark.py`. Las decisiones de ingeniería de cada nivel están en la guía; aquí se documentan en las celdas de texto marcadas como **Su decisión**. Las celdas de código usan la sesión `spark` del sub-paso B: cuando un script de `pipeline/` termina, cierra la sesión, y las celdas siguientes la abren de nuevo con `abrir_sesion()`. Si reinician el núcleo, vuelvan a ejecutar la configuración común, la celda de ayudantes y las celdas del Nivel 1 que definen `ESQUEMA` (C.1) y `MODOS` y `filas_y_suma` (C.2).

**Entorno.** Perfil `s16` del paquete en marcha (`docker compose --profile s16 up -d`, servicio `healthy`), con este cuaderno guardado en la carpeta `practica/` y abierto desde JupyterLab (http://localhost:8888). Python 3.11, Java 11, PySpark 3.5.3, pandas y pyarrow ya vienen en la imagen. Semilla fija `SEED = 42`. La sesión usa `local[4]` y `spark.driver.memory=1g`; el pico medido de todo el cuaderno en un mismo núcleo es de 2,4 a 2,7 GiB dentro del contenedor, que está limitado a 3 GiB (de 0,3 a 0,6 GiB de margen), y el de los scripts por terminal, de 1,1 a 1,3 GiB: abran una sola sesión de Spark a la vez. Los tiempos de reloj son orientativos: varían hasta ×3 según la carga del equipo; lo estable son los conteos, los trabajos, los bytes y los hashes. La interfaz de Spark se abre en http://localhost:4040 mientras la sesión esté viva.

**La regla de la práctica.** Un error de esquema no avisa: ninguna capa se declara correcta sin contar nulos por columna y sin reconciliarla, **sin filtro de calidad**, contra una referencia independiente; y ninguna decisión entre una expresión nativa y una función propia se declara justificada con una sola corrida.

### 0. Configuración común

La celda localiza el paquete, importa el script de la capa (`pipeline/refinada_spark.py`) para reutilizar sus funciones ya resueltas (sesión, firma por grupo, reconciliación, interfaz de Spark) y calcula el hash del `datalake/` heredado: al terminar, esa carpeta debe seguir idéntica.

In [1]:
# Configuracion comun: importaciones, semilla, rutas del paquete y ayudantes
import contextlib
import datetime as dt
import hashlib
import importlib
import inspect
import io
import json
import math
import os
import re
import shutil
import statistics
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType, TimestampType

SEED = 42                                          # semilla fija del proyecto (la usan los generadores de datos)
EQUIPO = os.environ.get("EQUIPO", "equipo-1")      # cambien el valor por el identificador de su equipo (equipo-1 a equipo-6)
DRIVER_MEMORY = "1g"                               # memoria de la JVM del controlador

# Raiz del paquete: la carpeta que contiene datalake/ y pipeline/. Este cuaderno vive en practica/.
for candidata in (Path.cwd(), Path.cwd().parent):
    if (candidata / "datalake").is_dir() and (candidata / "pipeline").is_dir():
        RAIZ = candidata
        break
else:
    raise FileNotFoundError("No encuentro datalake/ y pipeline/: abran este cuaderno desde la carpeta practica/ del paquete")
sys.path.insert(0, str(RAIZ))
from pipeline import refinada_spark as R    # el script de la capa: de el se reutilizan la sesion, la firma y la reconciliacion

UMBRAL_BAJA, UMBRAL_ALTA = R.UMBRAL_BAJA, R.UMBRAL_ALTA    # los mismos nombres que usa el script: las funciones se pueden copiar a pipeline/refinada_spark.py sin cambios
LAGO = RAIZ / "datalake"
CRUDA = LAGO / "cruda" / "acueducto"
HEREDADA = LAGO / "refinada" / "acueducto"


def hash_arbol(carpeta, excluir=("consolidada_spark", "refinada_spark")):
    """SHA-256 de un arbol de archivos: combina cada ruta relativa con el hash de su contenido, en orden.
    Sirve para comprobar que una carpeta NO cambio. Devuelve (hash, numero de archivos)."""
    h, n = hashlib.sha256(), 0
    for archivo in sorted(Path(carpeta).rglob("*")):
        relativa = archivo.relative_to(carpeta).as_posix()
        if archivo.is_file() and not relativa.startswith(excluir) and not archivo.name.endswith(".crc"):
            h.update(relativa.encode())
            h.update(hashlib.sha256(archivo.read_bytes()).hexdigest().encode())
            n += 1
    return h.hexdigest(), n


# Ayudantes que leen la interfaz de Spark por su API REST (la misma que alimenta las pestanas Jobs, Stages, Storage y Executors).
# Usan la variable global "spark": se asigna al abrir la sesion.
def abrir_sesion():
    """Abre la sesion de Spark del proyecto (o recupera la que sigue abierta): local[4], UTC y memoria de 1g."""
    global spark
    spark = R.crear_sesion("local[4]", DRIVER_MEMORY)
    return spark


def api(ruta):
    return R._rest(spark, ruta)


def marca():
    """(siguiente trabajo, siguiente etapa): sirve para contar solo lo que ocurre despues."""
    return R.marca_interfaz(spark)


def actividad(desde):
    """Trabajos, tareas, etapas y MiB leidos desde una marca."""
    time.sleep(0.5)    # la interfaz recibe los eventos de forma asincrona: se espera medio segundo
    j0, e0 = desde
    trabajos = sorted((j for j in api("jobs") if j["jobId"] >= j0), key=lambda j: j["jobId"])
    etapas = [e for e in api("stages") if e["stageId"] >= e0]
    completas = [e for e in etapas if e["status"] == "COMPLETE"]
    return {"trabajos": len(trabajos), "tareas_por_trabajo": [j["numCompletedTasks"] for j in trabajos],
            "etapas": len(completas), "etapas_omitidas": sum(1 for e in etapas if e["status"] == "SKIPPED"),
            "tareas": sum(e["numCompleteTasks"] for e in completas),
            "mib_leidos": round(sum(e["inputBytes"] for e in completas) / 2**20, 2),
            "mezcla_escrita_b": sum(e["shuffleWriteBytes"] for e in completas)}


def intentar(accion):
    """Ejecuta la accion; si Spark falla, devuelve el nombre de la causa en lugar de detener la celda."""
    try:
        return accion()
    except Exception as error:
        texto = str(error)
        for pista in ("MALFORMED_RECORD_IN_PARSING", "CSV header does not conform", "ArrowInvalid", "ArrowTypeError"):
            if pista in texto:
                return "ERROR: " + pista
        causas = re.findall(r"Caused by: ([\w.]+)", texto)
        return "ERROR: " + (causas[-1].split(".")[-1] if causas else type(error).__name__)


print("Raiz del paquete:", RAIZ)
print("Semilla:", SEED, "· equipo:", EQUIPO, "· memoria del controlador:", DRIVER_MEMORY)
datalake_antes, archivos_antes = hash_arbol(LAGO)
print(f"datalake/ heredado: {archivos_antes} archivos · SHA-256 {datalake_antes[:16]}...")

Raiz del paquete: /proyecto
Semilla: 42 · equipo: equipo-1 · memoria del controlador: 1g
datalake/ heredado: 360 archivos · SHA-256 0a986ffec04ad666...


**Ayudantes.** Tres funciones pequeñas que se usan más adelante: `tipo_marca_temporal` lee con pyarrow (sin Spark) el tipo físico y lógico de la marca temporal de un archivo Parquet, `entero_de_la_epoca` lee el entero que guarda el archivo para una lectura, y `tabla_resumen` arma la tabla de mediana, rango y razón frente a la nativa. Si reinician el núcleo, ejecuten de nuevo esta celda y la de configuración.

In [2]:
# Ayudantes para leer resultados: el tipo de la marca temporal con pyarrow (sin Spark) y la tabla resumen de los tiempos
def tipo_marca_temporal(ruta):
    esquema = pq.ParquetFile(ruta).schema
    columna = esquema.column(esquema.names.index("timestamp"))
    logico = re.search(r"isAdjustedToUTC=(\w+), timeUnit=(\w+)", str(columna.logical_type))
    return {"fisico": str(columna.physical_type),
            "logico": f"isAdjustedToUTC={logico.group(1)}, unidad={logico.group(2)}" if logico else str(columna.logical_type),
            "tipo_arrow": str(pq.read_table(ruta).schema.field("timestamp").type)}


def entero_de_la_epoca(ruta, medidor="MED0001", posicion=13):
    """El entero guardado para la lectura de las 13:00 del medidor, tal como esta en el archivo."""
    t = pq.read_table(ruta, columns=["meter_id", "timestamp"])
    enteros = t.column("timestamp").cast(pa.int64()).to_pylist()
    return [e for e, m in zip(enteros, t.column("meter_id").to_pylist()) if m == medidor][posicion]


def tabla_resumen(medidas):
    """Mediana (minimo a maximo), razon frente a la nativa y si el rango se solapa con el de la nativa."""
    nativa = medidas["nativa"]
    filas = []
    for v, m in medidas.items():
        solapa = None if v == "nativa" else not (m["min_s"] > nativa["max_s"] or m["max_s"] < nativa["min_s"])
        filas.append({"variante": v, "mediana_s": m["mediana_s"], "rango_s": f"{m['min_s']} a {m['max_s']}",
                      "razon_vs_nativa": round(m["mediana_s"] / nativa["mediana_s"], 1), "rango_se_solapa_con_nativa": solapa})
    return pd.DataFrame(filas).set_index("variante")

## Nivel 1 · Guiado · 45 minutos

Se resuelve sobre la **base común** (200 medidores, 120 días, 576 000 lecturas): la fuente es la capa cruda en CSV; la refinada heredada en Parquet es la **referencia** contra la que se reconcilia. Las celdas de este nivel ocupan unos 2 minutos de ejecución; el resto del tiempo es lectura de la interfaz y decisión. Si no alcanzan a escribir el contraste del sub-paso A o las celdas **Su decisión**, pueden diferirlos al comienzo del Nivel 2.

### Sub-paso A · Contraste con la solución de referencia de la carga con Spark (4 minutos)

Las dos celdas siguientes imprimen lo que el código de referencia (`pipeline/carga_spark.py`) declara de sí mismo y lo que registró su bitácora. Con esa lectura y con su registro de decisiones de la práctica anterior, completen en su repositorio la tabla de contraste de la guía: una fila por decisión, con lo que mantienen, lo que cambiarían y por qué. Una decisión distinta de la de referencia no es incorrecta si está bien justificada.

In [3]:
# Sub-paso A: lean lo que la solucion de referencia de la carga con Spark dice de si misma
import ast
from pipeline import carga_spark as cs

arbol_referencia = ast.parse((RAIZ / "pipeline" / "carga_spark.py").read_text(encoding="utf-8"))
print(ast.get_docstring(arbol_referencia))

Primera carga con Spark: agregado diario por sector de la capa refinada del lago.

Este archivo es un ESQUELETO: leer_refinada, agregar y escribir_salida estan por
completar (busquen TODO). La sesion de Spark (crear_sesion), los parametros, la lectura de
la interfaz de Spark, el hash de la salida, la reconciliacion y la bitacora ya funcionan;
el resto del texto de este encabezado describe el programa terminado.

Que hace
    Lee la capa refinada (Parquet) del lago del acueducto, calcula el consumo total y el
    numero de lecturas por dia y sector, y escribe el resultado en Parquet:
      1. Crea una sesion local de Spark (local[N]) con la zona horaria fijada en UTC.
      2. Lee la carpeta refinada completa. La marca temporal viene guardada en
         NANOSEGUNDOS y Spark 3.5 no la lee como marca temporal: se pide como entero
         largo (spark.sql.legacy.parquet.nanosAsLong=true) y se convierte a segundos.
      3. Agrega por (fecha, sector) con reduceByKey (combina dentro de cad

In [4]:
# Sub-paso A (continuacion): las funciones donde viven dos de las decisiones, y lo que registro la bitacora de referencia
print(inspect.getdoc(cs.agregar))
print()
print(inspect.getdoc(cs.escribir_salida))
bitacora_s15 = pd.read_json(RAIZ / "bitacora_carga_spark.jsonl", lines=True)
print()
print(bitacora_s15[["ruta", "metodo", "particiones_entrada", "tareas", "mezcla_escritura_bytes", "tiempo_s", "reconciliacion"]].to_string(index=False))

Suma el consumo y cuenta las lecturas por (fecha, sector).

Devuelve un RDD de tuplas (fecha, sector, consumo_total, n_lecturas).

TODO (equipo): implementen las dos opciones de metodo.
  - "reduceByKey": combina dentro de cada particion antes de mezclar. Piensen que
    valor viaja por clave para poder sumar el consumo Y contar las lecturas.
  - "groupByKey": mezcla todos los valores y suma despues.
Ambas reciben particiones_mezcla (None = por omision). Justifiquen en su registro de
decisiones por que reduceByKey es valido aqui (funcion asociativa y conmutativa).

Con reduceByKey viaja por clave el par (suma_consumo, n_lecturas): combinar dos
parciales (a1, n1) y (a2, n2) en (a1 + a2, n1 + n2) es asociativo y conmutativo, asi
que Spark puede combinar dentro de cada particion antes de mezclar (combinador del
lado del mapa) y en cualquier orden despues. groupByKey, en cambio, mezcla cada
consumo individual y suma al final.

Escribe el resultado en carpeta_salida de forma que repetir la 

**Su decisión · Sub-paso A (equipo-1).** Contraste de nuestra T15 con la referencia `pipeline/carga_spark.py`:

| Decisión | Referencia | Nuestra T15 | Qué mantenemos o cambiaríamos hoy |
|---|---|---|---|
| Agregación con combinación previa | `reduceByKey` | `reduceByKey`: 9 822 B frente a 3 240 135 B con `groupByKey` en la ampliación | **Coincide.** La mantenemos para el RDD. Hoy la contrastamos con el `groupBy` de DataFrame (N2.8). |
| Reconciliación sin filtro | Sin filtro de calidad | Sin filtro: 480 de 480 grupos | **Coincide.** Hoy la capa conserva las lecturas no confiables y las marca con `es_confiable`. |
| Formato de entrada | La refinada en Parquet | La refinada en Parquet (CSV: 30 veces las tareas) | Hoy cambia por diseño: la capa nueva **se construye desde la cruda**, con esquema declarado. La refinada heredada pasa a ser la referencia. |
| Sobrescritura completa, un solo archivo | `overwrite` con `coalesce(1)` | Igual | Para 480 filas **coincide**. Para la capa de 576 000 a 1 002 240 filas cambiamos a una partición por día con sobrescritura **dinámica**, para reprocesar una ventana sin reescribir los 120 días. |


### Sub-paso B · Levantar el perfil `s16` y abrir la interfaz de Spark (3 minutos)

Desde la terminal del computador: `docker compose --profile s12 down`, luego `docker compose --profile s16 up -d`. `s16` es **el mismo servicio** `spark` que la práctica anterior con una etiqueta de perfil más: el contenedor sigue llamándose `acueducto_spark_s15` y la imagen `acueducto-spark:s15`; los comandos se dirigen al servicio (`exec spark`), no al nombre del contenedor. En la sesión `local[4]` todo vive en una sola JVM: el único "ejecutor" es el propio controlador, y la memoria de almacenamiento sale de `spark.driver.memory`: $M=(H-300\,\text{MiB})\cdot 0{,}6$ con $H=1024$ MiB.

In [5]:
# B.1 Comprobacion del entorno: que versiones corren dentro del contenedor
import platform
import pyspark

print("Python", platform.python_version(), "· PySpark", pyspark.__version__, "· pandas", pd.__version__, "· pyarrow", pa.__version__)
print("Java", re.search(r'version "([^"]+)"', subprocess.run(["java", "-version"], capture_output=True, text=True).stderr).group(1))
print("Nucleos visibles para el contenedor:", os.cpu_count())

Python 3.11.17 · PySpark 3.5.3 · pandas 2.2.2 · pyarrow 15.0.2
Java 11.0.32.1
Nucleos visibles para el contenedor: 6


In [6]:
# B.2 La sesion de Spark: local[4] = una sola JVM con 4 ranuras de tareas; crear_sesion ya fija UTC y la memoria
spark = abrir_sesion()
sc = spark.sparkContext
print("Spark", spark.version, "· master:", sc.master, "· paralelismo por omision:", sc.defaultParallelism)
print("Interfaz de Spark: http://localhost:" + sc.uiWebUrl.rsplit(":", 1)[-1])
print("Memoria del controlador:", sc.getConf().get("spark.driver.memory"), "· zona horaria de la sesion:", spark.conf.get("spark.sql.session.timeZone"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 22:04:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 · master: local[4] · paralelismo por omision: 4
Interfaz de Spark: http://localhost:4040


Memoria del controlador: 1g · zona horaria de la sesion: UTC


In [7]:
# B.3 Pestana Executors (por su API): en modo local hay UN solo ejecutor, llamado "driver"
ejecutores = api("allexecutors")
for e in ejecutores:
    print(f"ejecutor {e['id']}: {e['totalCores']} ranuras · memoria de almacenamiento maxima {e['maxMemory'] / 2**20:.1f} MiB")
maxima_mib = round(ejecutores[0]["maxMemory"] / 2**20, 1)
print("Formula M = (H - 300 MiB) * 0,6 con H = 1024 MiB:", round((1024 - 300) * 0.6, 1), "MiB")

ejecutor driver: 4 ranuras · memoria de almacenamiento maxima 434.4 MiB
Formula M = (H - 300 MiB) * 0,6 con H = 1024 MiB: 434.4 MiB


In [8]:
# Autoverificacion del sub-paso B
assert len(ejecutores) == 1 and ejecutores[0]["id"] == "driver", "en modo local hay un unico ejecutor: el controlador"
assert ejecutores[0]["totalCores"] == 4, "local[4] ofrece 4 ranuras de tareas"
assert spark.conf.get("spark.sql.session.timeZone") == "UTC", "la zona horaria de la sesion debe ser UTC"
if DRIVER_MEMORY == "1g":
    assert abs(maxima_mib - 434.4) < 0.2, "con 1g, la memoria de almacenamiento es (1024 - 300) * 0,6 = 434,4 MiB"
print("Autoverificacion del sub-paso B superada: un ejecutor, 4 ranuras, UTC y", maxima_mib, "MiB de almacenamiento.")

Autoverificacion del sub-paso B superada: un ejecutor, 4 ranuras, UTC y 434.4 MiB de almacenamiento.


**Su decisión · Sub-paso B.** Mantenemos `local[4]` con `1g`. Executors muestra **1 ejecutor (`driver`), 4 ranuras y 434,4 MiB** de memoria de almacenamiento. En la T15, el pico medido con `docker stats` fue de 2,19 GiB de 3 GiB, y en esta sesión de 2,30 GiB: con 2g, el montón crecería 1 024 MiB y superaría el tope del contenedor. `s16` no crea otro contenedor porque el servicio `spark` pertenece a los perfiles s15 a s18, con la misma imagen (`acueducto-spark:s15`) y el mismo nombre de contenedor. Solo cambia el perfil con que se levanta.

### Sub-paso C · La cruda en CSV con el esquema inferido, declarado y mal declarado (11 minutos)

**C.1** compara tres lecturas de la misma carpeta: con `inferSchema`, sin inferir y sin esquema (todo texto) y con el esquema declarado. La inferencia cuesta trabajos y MiB leídos; el esquema declarado no lanza ningún trabajo al leer. **C.2** declara `consumo_m3` como entero por error y mira lo que ocurre con `count()`, con una suma y con cada modo de lectura. **C.3** quita el encabezado y cambia el orden de dos columnas en un archivo: la opción por omisión de `enforceSchema` deja pasar el cambio sin error ni nulos.

In [9]:
# C.1 Esquema inferido, sin inferir y declarado: cuantos trabajos lanza la LECTURA y cuantos MiB se leen
ESQUEMA = StructType([
    StructField("meter_id", StringType(), True),       # identificador del medidor
    StructField("timestamp", TimestampType(), True),   # texto 'AAAA-MM-DD HH:MM:SS' sin zona, leido como marca temporal
    StructField("sector", StringType(), True),
    StructField("consumo_m3", DoubleType(), True),     # double: los valores traen punto decimal
    StructField("presion_bar", DoubleType(), True),
    StructField("quality_flag", StringType(), True),
])


def leer_y_contar(etiqueta, **opciones):
    """Lee la cruda con las opciones dadas y mide en la interfaz de Spark la LECTURA y despues un count()."""
    m = marca()
    df = spark.read.csv(str(CRUDA), header=True, **opciones)
    leer = actividad(m)
    m = marca()
    filas = df.count()
    contar = actividad(m)
    return df, {"lectura": etiqueta, "trabajos_al_leer": leer["trabajos"], "tareas_por_trabajo": leer["tareas_por_trabajo"],
                "mib_al_leer": leer["mib_leidos"], "mib_al_contar": contar["mib_leidos"],
                "mib_total": round(leer["mib_leidos"] + contar["mib_leidos"], 2), "filas": filas}


df_inferido, f_inferida = leer_y_contar("inferida", inferSchema=True)
df_texto, f_texto = leer_y_contar("sin inferir ni esquema")
df_declarado, f_declarada = leer_y_contar("declarada", schema=ESQUEMA)
comparacion_lecturas = pd.DataFrame([f_inferida, f_texto, f_declarada]).set_index("lectura")
print(comparacion_lecturas.to_string())

                        trabajos_al_leer tareas_por_trabajo  mib_al_leer  mib_al_contar  mib_total   filas
lectura                                                                                                   
inferida                               3        [120, 1, 4]        28.30          28.24      56.54  576000
sin inferir ni esquema                 2           [120, 1]         0.06          28.24      28.30  576000
declarada                              0                 []         0.00          28.24      28.24  576000


In [10]:
# C.1 (continuacion) Los tipos que resultan de cada lectura
print("Inferidos:  ", dict(df_inferido.dtypes))
print("Sin inferir:", dict(df_texto.dtypes))
print("Declarados: ", dict(df_declarado.dtypes))

Inferidos:   {'meter_id': 'string', 'timestamp': 'timestamp', 'sector': 'string', 'consumo_m3': 'double', 'presion_bar': 'double', 'quality_flag': 'string', 'anio': 'int', 'mes': 'int', 'dia': 'int'}
Sin inferir: {'meter_id': 'string', 'timestamp': 'string', 'sector': 'string', 'consumo_m3': 'string', 'presion_bar': 'string', 'quality_flag': 'string', 'anio': 'int', 'mes': 'int', 'dia': 'int'}
Declarados:  {'meter_id': 'string', 'timestamp': 'timestamp', 'sector': 'string', 'consumo_m3': 'double', 'presion_bar': 'double', 'quality_flag': 'string', 'anio': 'int', 'mes': 'int', 'dia': 'int'}


In [11]:
# C.2 consumo_m3 declarado como ENTERO por error: tres modos de lectura y dos formas de mirar el resultado
ESQUEMA_ENTERO = StructType([StructField(f.name, IntegerType() if f.name == "consumo_m3" else f.dataType, True) for f in ESQUEMA.fields])
MODOS = ("PERMISSIVE", "DROPMALFORMED", "FAILFAST")
lecturas_modo = {modo: spark.read.csv(str(CRUDA), header=True, schema=ESQUEMA_ENTERO, mode=modo) for modo in MODOS}


def filas_y_suma(df):
    """Cuenta filas y suma el consumo EN LA MISMA agregacion: obliga a Spark a convertir la columna consumo_m3."""
    fila = df.agg(F.count("*").alias("filas"), F.sum("consumo_m3").alias("suma")).collect()[0]
    return fila["filas"], fila["suma"]


tabla_modos = pd.DataFrame([{"modo": modo, "count()": intentar(df.count), "filas y suma de consumo_m3": intentar(lambda df=df: filas_y_suma(df))}
                            for modo, df in lecturas_modo.items()])
print(tabla_modos.to_string(index=False))

         modo  count()         filas y suma de consumo_m3
   PERMISSIVE   576000                     (576000, None)
DROPMALFORMED   576000                          (0, None)
     FAILFAST   576000 ERROR: MALFORMED_RECORD_IN_PARSING


26/11/12 22:04:58 ERROR Executor: Exception in task 2.0 in stage 29.0 (TID 288)
org.apache.spark.SparkException: Encountered error while reading file file:///proyecto/datalake/cruda/acueducto/anio=2024/mes=04/dia=13/lecturas.csv. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:864)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:293)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithoutKey_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.ex

In [12]:
# C.2 (continuacion) Nulos por columna con PERMISSIVE y reconciliacion por grupo contra consolidada/
calidad_entero = R.resumen_calidad(lecturas_modo["PERMISSIVE"])
print("Filas:", calidad_entero["filas"], "· nulos por columna:", calidad_entero["nulos"])

referencia = R.referencia_consolidada(LAGO)        # 480 filas: fecha, sector, consumo total y lecturas
por_grupo = (lecturas_modo["PERMISSIVE"].groupBy(F.to_date("timestamp").cast("string").alias("fecha"), "sector")
             .agg(F.sum("consumo_m3").alias("suma"), F.count("*").alias("n")).toPandas())
cotejo = por_grupo.merge(referencia, on=["fecha", "sector"])
grupos_que_coinciden = int(((cotejo["suma"] - cotejo["consumo_total_m3"]).abs() <= 1e-6).sum())
print(f"Grupos (fecha, sector): {len(cotejo)} · con la suma igual a consolidada/: {grupos_que_coinciden}")

Filas: 576000 · nulos por columna: {'meter_id': 0, 'timestamp': 0, 'sector': 0, 'consumo_m3': 576000, 'presion_bar': 0, 'quality_flag': 0}


Grupos (fecha, sector): 480 · con la suma igual a consolidada/: 0


In [13]:
# C.3 (a) Encabezado no declarado (header=False): la primera linea de cada archivo entra como un dato
sin_encabezado = spark.read.csv(str(CRUDA), header=False, schema=ESQUEMA)
filas_sin_encabezado = sin_encabezado.count()
valores_sector = sin_encabezado.groupBy("sector").count().orderBy("count").toPandas()
calidad_sin_encabezado = R.resumen_calidad(sin_encabezado)
print("Filas:", filas_sin_encabezado, "(esperadas 576 000)")
print(valores_sector.to_string(index=False))
print("Nulos por columna con el esquema declarado:", calidad_sin_encabezado["nulos"])

Filas: 576120 (esperadas 576 000)
 sector  count
 sector    120
S-NORTE 144000
  S-SUR 144000
S-OESTE 144000
 S-ESTE 144000
Nulos por columna con el esquema declarado: {'meter_id': 0, 'timestamp': 120, 'sector': 0, 'consumo_m3': 120, 'presion_bar': 120, 'quality_flag': 0}


In [14]:
# C.3 (b) Orden de columnas distinto en UN archivo: dos columnas double intercambiadas no dan nulos ni error con la opcion por omision
TMP_C3 = Path(tempfile.mkdtemp(prefix="c3_"))
for dia in ("04", "05"):
    destino = TMP_C3 / "anio=2024" / "mes=01" / f"dia={dia}"
    destino.mkdir(parents=True)
    origen = CRUDA / "anio=2024" / "mes=01" / f"dia={dia}" / "lecturas.csv"
    if dia == "05":    # copia temporal con presion_bar ANTES de consumo_m3
        columnas_cambiadas = ["meter_id", "timestamp", "sector", "presion_bar", "consumo_m3", "quality_flag"]
        pd.read_csv(origen)[columnas_cambiadas].to_csv(destino / "lecturas.csv", index=False, lineterminator="\r\n")
    else:
        shutil.copy(origen, destino / "lecturas.csv")


def consumo_por_sector(enforce_schema):
    """Consumo del 2024-01-05 por sector, leyendo la copia con enforceSchema verdadero o falso."""
    df = spark.read.csv(str(TMP_C3), header=True, schema=ESQUEMA, enforceSchema=enforce_schema)
    filas = (df.filter(F.to_date("timestamp") == F.lit("2024-01-05").cast("date")).groupBy("sector")
               .agg(F.round(F.sum("consumo_m3"), 1).alias("consumo_m3")).orderBy("sector").collect())
    return {f["sector"]: f["consumo_m3"] for f in filas}


dia_ref = referencia[referencia["fecha"] == "2024-01-05"].set_index("sector")["consumo_total_m3"].round(1)
con_omision = consumo_por_sector(True)
con_falso = intentar(lambda: consumo_por_sector(False))
cuentas_orden = {valor: spark.read.csv(str(TMP_C3), header=True, schema=ESQUEMA, enforceSchema=valor).count() for valor in (True, False)}
tabla_orden = pd.DataFrame({"consolidada/": dia_ref, "enforceSchema=True (omision)": pd.Series(con_omision),
                            "enforceSchema=False": pd.Series(con_falso if isinstance(con_falso, dict) else {s: con_falso for s in dia_ref.index})})
print(tabla_orden.to_string())
print("count() de la copia, con True y con False:", cuentas_orden)
shutil.rmtree(TMP_C3)    # se borra la copia temporal

26/11/12 22:05:03 ERROR Executor: Exception in task 0.0 in stage 58.0 (TID 322)
java.lang.IllegalArgumentException: CSV header does not conform to the schema.
 Header: timestamp, sector, presion_bar
 Schema: timestamp, sector, consumo_m3
Expected: consumo_m3 but found: presion_bar
CSV file: file:///tmp/c3_v426tvbu/anio=2024/mes=01/dia=05/lecturas.csv
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2(CSVHeaderChecker.scala:95)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2$adapted(CSVHeaderChecker.scala:91)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.checkHeaderColumnNames(CSVHeaderChecker.scala:91)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$6(CSVHeaderChecker.scala:127)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$6$adapted(CSVHeaderChecker.scala:126)
	at scala.Option.

         consolidada/  enforceSchema=True (omision)                 enforceSchema=False
S-ESTE          190.7                        3601.5  ERROR: CSV header does not conform
S-NORTE         193.0                        3614.3  ERROR: CSV header does not conform
S-OESTE         193.0                        3600.0  ERROR: CSV header does not conform
S-SUR           191.9                        3599.3  ERROR: CSV header does not conform
count() de la copia, con True y con False: {True: 9600, False: 9600}


In [15]:
# Autoverificacion del sub-paso C: el comportamiento manda; las cifras medidas se informan
assert f_inferida["trabajos_al_leer"] > f_texto["trabajos_al_leer"] > f_declarada["trabajos_al_leer"] == 0, "inferir lanza mas trabajos que no inferir; declarar, ninguno"
assert f_inferida["mib_total"] > 1.9 * f_declarada["mib_total"], "inferir lee los datos dos veces: un esquema declarado, una"
assert dict(df_inferido.dtypes)["timestamp"] == "timestamp" and dict(df_texto.dtypes)["consumo_m3"] == "string"
assert [r["count()"] for r in tabla_modos.to_dict("records")] == [576_000] * 3, "count() no detecta nada en ninguno de los tres modos"
assert calidad_entero["nulos"]["consumo_m3"] == 576_000 and sum(calidad_entero["nulos"].values()) == 576_000, "todo el consumo quedo nulo y nada mas"
assert grupos_que_coinciden == 0, "con el consumo nulo ninguno de los 480 grupos reconcilia"
assert filas_sin_encabezado == 576_120 and calidad_sin_encabezado["nulos"]["timestamp"] == 120
assert con_omision["S-ESTE"] > 3_000 and con_omision["S-ESTE"] > 10 * float(dia_ref["S-ESTE"]), "con la omision, el consumo se leyo de la columna de presion"
assert cuentas_orden[True] == cuentas_orden[False] == 9_600, "count() no detecta el orden distinto, ni con True ni con False"
assert isinstance(con_falso, str) and "header does not conform" in con_falso, "con enforceSchema=False la agregacion falla"
print("Autoverificacion del sub-paso C superada: los errores de esquema son silenciosos y solo una prueba de nulos, de suma o de reconciliacion los muestra.")

Autoverificacion del sub-paso C superada: los errores de esquema son silenciosos y solo una prueba de nulos, de suma o de reconciliacion los muestra.


**Su decisión · Sub-paso C.** Cada error silencioso lo detecta una prueba distinta, y `count()` no detecta ninguno:
- **`consumo_m3` declarado como entero:** `count()` da 576 000, igual que siempre. Lo detectan los **nulos por columna** (576 000 nulos en `consumo_m3`) y la **reconciliación por grupo** (0 de 480 grupos con la suma igual a `consolidada/`).
- **Encabezado no declarado** (`header=False`): lo detecta el **conteo de filas frente al esperado** (576 120 frente a 576 000) o los 120 nulos en `timestamp` y en los números.
- **Columnas en otro orden:** con `enforceSchema=True` no hay nulos ni errores, pero la suma del sector se dispara (S-ESTE: 3 601,5 frente a 190,7 m³). Lo detecta la **reconciliación de sumas** o `enforceSchema=False`, que **falla** con "CSV header does not conform".

`enforceSchema=False` viaja resuelto en `leer_cruda` porque convierte el error más peligroso, datos en la columna equivocada sin ningún aviso, en un fallo visible de la lectura. Ese contrato no debe depender de que alguien se acuerde de activarlo.

### Sub-paso D · Construir `refinada_spark/` y reconciliarla (12 minutos)

**D.1** y **D.2** leen la cruda con el esquema declarado y derivan `fecha`, `hora`, `es_confiable`, `anio`, `mes`, `dia` y `nivel_presion`; **`anio`, `mes` y `dia` se derivan del instante como texto con ceros** (la convención del lago es `mes=01`, no `mes=1`) y la capa conserva todas las filas. **D.3** escribe Parquet con la marca temporal en microsegundos, partición por día y sobrescritura dinámica. **D.4** lee el resultado con pyarrow (tipo físico y lógico de la marca temporal, el de la capa heredada y el de la nueva) y reconcilia la capa contra la refinada heredada (leída con `nanosAsLong`) y contra `consolidada/`.

In [16]:
# D.1 La lectura definitiva: esquema declarado, modo PERMISSIVE, formato de fecha explicito y enforceSchema=False
print(inspect.getsource(R.leer_cruda))
crudo = R.leer_cruda(spark, CRUDA, ESQUEMA)
particiones_crudo = crudo.rdd.getNumPartitions()
print("Columnas:", crudo.columns, "· particiones:", particiones_crudo)

def leer_cruda(spark, carpeta_cruda, esquema):
    """Lee la carpeta cruda completa con el esquema declarado y modo PERMISSIVE (un valor que no
    convierte queda nulo; no detiene la lectura). El formato de la marca temporal se declara
    de forma explicita. Con enforceSchema=False Spark compara el encabezado de cada archivo con el
    esquema declarado, por posicion: si una fuente cambia el orden de las columnas, la lectura falla
    en lugar de asignar los tipos a la columna equivocada (con el valor por omision, True, el
    encabezado se ignora y los datos quedan en la columna equivocada sin ningun error). Al leer la carpeta, Spark agrega anio, mes y dia como enteros tomados de
    la ruta: se descartan, porque la verdad esta en la marca temporal y se deriva de ella."""
    df = (spark.read.csv(str(carpeta_cruda), header=True, schema=esquema, mode="PERMISSIVE",
                         timestampFormat="yyyy-MM-dd HH:mm:ss", enforceSchema=False))
    return df.drop("anio", "mes", "

In [17]:
# D.2 Columnas derivadas y clasificacion nativa. anio, mes y dia se derivan del instante, como TEXTO con ceros
def derivar_columnas(df):
    """Agrega fecha, hora, es_confiable y las tres partes de la ruta. La capa CONSERVA todas las filas."""
    return (df.withColumn("fecha", F.to_date("timestamp"))
              .withColumn("hora", F.hour("timestamp"))
              .withColumn("es_confiable", F.col("quality_flag") == "OK")
              .withColumn("anio", F.date_format("timestamp", "yyyy"))
              .withColumn("mes", F.date_format("timestamp", "MM"))       # 'MM' da '01'; un entero daria 1
              .withColumn("dia", F.date_format("timestamp", "dd")))


def clasificar_presion(presion):
    """Expresion nativa: se compila y corre dentro de la JVM. Una presion nula da clase nula."""
    return (F.when(presion < UMBRAL_BAJA, "baja")
             .when(presion > UMBRAL_ALTA, "alta")
             .when(presion.isNotNull(), "normal"))


capa_n1 = derivar_columnas(crudo).withColumn("nivel_presion", clasificar_presion(F.col("presion_bar")))
capa_n1.printSchema()
calidad_capa = R.resumen_calidad(capa_n1)
confiables = capa_n1.agg(F.sum(F.col("es_confiable").cast("int"))).collect()[0][0]
print("Filas:", calidad_capa["filas"], "· filas con algun nulo en las seis columnas:", calidad_capa["no_conformes"])
print("Nulos por columna:", calidad_capa["nulos"])
print("es_confiable verdadero:", confiables, "· falso:", calidad_capa["filas"] - confiables)
print("Columnas que Spark agrega al leer la carpeta (de la ruta), antes de derivar:",
      {c: t for c, t in spark.read.csv(str(CRUDA), header=True, schema=ESQUEMA).dtypes if c in ("anio", "mes", "dia")})

root
 |-- meter_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- sector: string (nullable = true)
 |-- consumo_m3: double (nullable = true)
 |-- presion_bar: double (nullable = true)
 |-- quality_flag: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- hora: integer (nullable = true)
 |-- es_confiable: boolean (nullable = true)
 |-- anio: string (nullable = true)
 |-- mes: string (nullable = true)
 |-- dia: string (nullable = true)
 |-- nivel_presion: string (nullable = true)



Filas: 576000 · filas con algun nulo en las seis columnas: 0
Nulos por columna: {'meter_id': 0, 'timestamp': 0, 'sector': 0, 'consumo_m3': 0, 'presion_bar': 0, 'quality_flag': 0}
es_confiable verdadero: 541368 · falso: 34632
Columnas que Spark agrega al leer la carpeta (de la ruta), antes de derivar: {'anio': 'int', 'mes': 'int', 'dia': 'int'}


In [18]:
# D.3 Escribir la capa: Parquet con la marca temporal en MICROSEGUNDOS, particionado por dia, sobrescritura dinamica
def guardar_capa(df, ruta, sobrescritura="dinamica"):
    """Escribe la capa. El valor por omision de la marca temporal es INT96, que otros lectores interpretan distinto."""
    df.sparkSession.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
    df.sparkSession.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic" if sobrescritura == "dinamica" else "static")
    df.write.mode("overwrite").partitionBy("anio", "mes", "dia").parquet(str(ruta))


SALIDA_N1 = LAGO / "refinada_spark"
t0 = time.perf_counter()
guardar_capa(capa_n1, SALIDA_N1)
tiempo_escritura = round(time.perf_counter() - t0, 1)

arbol_n1 = R.resumen_arbol(SALIDA_N1)
archivos_capa = sorted(SALIDA_N1.rglob("*.parquet"))
bytes_capa = sum(a.stat().st_size for a in archivos_capa)
peso_mb = round(bytes_capa / 1e6, 3)
kib_por_archivo = round(bytes_capa / len(archivos_capa) / 1024, 1)
print(f"Carpetas de dia: {len(arbol_n1)} · archivos: {len(archivos_capa)} · {peso_mb} MB · {kib_por_archivo} KiB por archivo · escritura: {tiempo_escritura} s")
print("Una carpeta del arbol:", next(iter(arbol_n1)), "· archivos part-*.parquet dentro:", len(next(iter(arbol_n1.values()))))

Carpetas de dia: 120 · archivos: 120 · 3.006 MB · 24.5 KiB por archivo · escritura: 3.2 s
Una carpeta del arbol: anio=2024/mes=01/dia=01 · archivos part-*.parquet dentro: 1


In [19]:
# D.4 (a) Lector independiente: pyarrow lee el pie del archivo Parquet, sin Spark. Tipo FISICO y tipo LOGICO de la marca temporal
print(inspect.getsource(tipo_marca_temporal))
archivo_nuevo = sorted(SALIDA_N1.glob("anio=2024/mes=01/dia=01/*.parquet"))[0]
archivo_heredado = HEREDADA / "anio=2024" / "mes=01" / "dia=01" / "lecturas.parquet"
tipos_marca = pd.DataFrame({"nueva (refinada_spark/)": tipo_marca_temporal(archivo_nuevo), "heredada (refinada/)": tipo_marca_temporal(archivo_heredado)})
print(tipos_marca.to_string())
entero_nuevo, entero_heredado = entero_de_la_epoca(archivo_nuevo), entero_de_la_epoca(archivo_heredado)
print(f"MED0001 a las 13:00 · nueva: {entero_nuevo} ({len(str(entero_nuevo))} digitos) · heredada: {entero_heredado} ({len(str(entero_heredado))} digitos)")

def tipo_marca_temporal(ruta):
    esquema = pq.ParquetFile(ruta).schema
    columna = esquema.column(esquema.names.index("timestamp"))
    logico = re.search(r"isAdjustedToUTC=(\w+), timeUnit=(\w+)", str(columna.logical_type))
    return {"fisico": str(columna.physical_type),
            "logico": f"isAdjustedToUTC={logico.group(1)}, unidad={logico.group(2)}" if logico else str(columna.logical_type),
            "tipo_arrow": str(pq.read_table(ruta).schema.field("timestamp").type)}

                              nueva (refinada_spark/)                       heredada (refinada/)
fisico                                          INT64                                      INT64
logico      isAdjustedToUTC=true, unidad=microseconds  isAdjustedToUTC=false, unidad=nanoseconds
tipo_arrow                      timestamp[us, tz=UTC]                              timestamp[ns]
MED0001 a las 13:00 · nueva: 1704114000000000 (16 digitos) · heredada: 1704114000000000000 (19 digitos)


In [20]:
# D.4 (b) Reconciliacion SIN filtro de calidad: firma por (fecha, sector, calidad) de la capa nueva, de la heredada y de consolidada/
firma_nueva = R.firma_capa_nueva(spark, SALIDA_N1)                 # relee la capa desde disco
firma_heredada = R.firma_heredada(spark, HEREDADA)                 # lee la heredada con nanosAsLong y divide entre 1 000
ok_heredada, detalle_heredada, dif_heredada = R.comparar_firmas(firma_nueva, firma_heredada)
ok_consolidada, detalle_consolidada = R.reconciliar_agregado(firma_nueva, R.referencia_consolidada(LAGO))
hash_capa = R.hash_firma(firma_nueva)
print("Contra refinada/ (heredada):", ok_heredada, "·", detalle_heredada)
print("Contra consolidada/:        ", ok_consolidada, "·", detalle_consolidada)
print("Hash del contenido:", hash_capa)

Contra refinada/ (heredada): True · 1440 grupos iguales (tolerancia 1e-06); diferencia maxima de sumas 0.00e+00
Contra consolidada/:         True · 480 grupos iguales a la referencia (tolerancia 1e-06)
Hash del contenido: 9cc455c0aff897a5a4a4a3c7e851152c19d2c623c9f9a0cb76c2e16e8ab37c7a


In [21]:
# Autoverificacion del sub-paso D: el comportamiento manda
assert calidad_capa["filas"] == 576_000 and calidad_capa["no_conformes"] == 0 and sum(calidad_capa["nulos"].values()) == 0
assert confiables == 541_368, "la capa conserva las 34 632 lecturas no confiables: el filtro es decision del consumo"
assert len(arbol_n1) == 120 and len(archivos_capa) == 120 and all(len(v) == 1 for v in arbol_n1.values()), "un archivo por dia"
assert all(re.search(r"anio=\d{4}/mes=\d{2}/dia=\d{2}$", k) for k in arbol_n1), "carpetas con ceros"
assert tipos_marca.loc["fisico", "nueva (refinada_spark/)"] == "INT64"
assert tipos_marca.loc["logico", "nueva (refinada_spark/)"] == "isAdjustedToUTC=true, unidad=microseconds"
assert tipos_marca.loc["logico", "heredada (refinada/)"] == "isAdjustedToUTC=false, unidad=nanoseconds"
assert entero_heredado == entero_nuevo * 1000 and len(str(entero_nuevo)) == 16, "microsegundos: 16 digitos; nanosegundos: 19"
assert ok_heredada and ok_consolidada, "la capa nueva reconcilia sin filtro con la heredada y con consolidada/"
assert dif_heredada <= 1e-6
assert len(firma_nueva) == 1_440 and int(firma_nueva["n"].sum()) == 576_000
print("Autoverificacion del sub-paso D superada: capa reconciliada con la heredada y con consolidada/, con la marca temporal en microsegundos.")

Autoverificacion del sub-paso D superada: capa reconciliada con la heredada y con consolidada/, con la marca temporal en microsegundos.


**Su decisión · Sub-paso D.** Usamos **microsegundos** (`TIMESTAMP_MICROS`) porque el valor por omisión, `INT96`, es un formato heredado que otros lectores interpretan distinto. Con `INT64` y tipo lógico `isAdjustedToUTC=true, unidad=microseconds`, pyarrow lee `timestamp[us, tz=UTC]` sin ambigüedad: MED0001 a las 13:00 es 1704114000000000 en la capa nueva y 1704114000000000000 en la heredada (×1 000, el mismo instante). `anio`, `mes` y `dia` son **texto** para que la ruta sea `mes=01` y no `mes=1`. En N2.6 lo comprobamos: con enteros, las carpetas `mes=1` conviven con `mes=01` y al releer hay **28 800 filas duplicadas**. La capa **conserva** las lecturas no confiables (576 000 filas, 541 368 con `es_confiable` verdadero) porque las referencias (`refinada/`, `consolidada/`, `totales_control.csv`) cuentan todas las lecturas. Filtrar en la capa haría imposible reconciliarla. Quien necesite solo las confiables las filtra al leer.

### Sub-paso E · Tres formas de clasificar la presión y qué cuesta cada una (12 minutos)

Las tres variantes dan el mismo conteo por clase; lo que cambia es **cuántas veces se invoca la función, cuántas filas viajan a un proceso de Python y en qué formato**. **E.1** compara el resultado; **E.2** aplica el protocolo de medición (1 corrida de calentamiento descartada y 3 medidas, con una acción que **consume** la clase) y muestra la trampa de construir la consulta una sola vez; **E.3** lee el nodo del plan que evalúa la función en Python; **E.4** cuenta invocaciones, filas serializadas y lotes de Arrow con acumuladores; **E.5** repite la medición con la columna de presión ya en memoria, para ver el costo de la función sin el de leer el CSV.

In [22]:
# E.1 Tres formas de escribir la MISMA clasificacion: nativa, funcion por fila (UDF) y funcion vectorizada (Pandas UDF)
VARIANTES_N1 = {
    "nativa": lambda: clasificar_presion(F.col("presion_bar")),
    "udf_fila": lambda: R.clasificar_udf_fila(F.col("presion_bar")),
    "udf_pandas": lambda: R.clasificar_udf_pandas(F.col("presion_bar")),
}
base_variantes = derivar_columnas(crudo)


def consulta_clase(variante, base=None):
    """Consulta que CONSUME la clase: cuenta las lecturas por nivel_presion. Devuelve una consulta NUEVA en cada llamada."""
    base = base_variantes if base is None else base
    return base.withColumn("nivel_presion", VARIANTES_N1[variante]()).groupBy("nivel_presion").count()


conteos = {v: {str(r["nivel_presion"]): r["count"] for r in consulta_clase(v).collect()} for v in VARIANTES_N1}
print(pd.DataFrame(conteos).loc[["baja", "normal", "alta"]].to_string())

        nativa  udf_fila  udf_pandas
baja     27563     27563       27563
normal  520965    520965      520965
alta     27472     27472       27472


In [23]:
# E.2 Protocolo de medicion: 1 corrida de calentamiento (se descarta) y 3 corridas medidas; mediana, minimo y maximo
def medir(construir, corridas=3):
    """construir() debe devolver una consulta NUEVA en cada llamada: con la ejecucion adaptativa activa, volver a ejecutar
    el MISMO objeto reutiliza la mezcla ya materializada y las corridas siguientes casi no hacen trabajo."""
    tiempos = []
    for _ in range(corridas + 1):
        consulta = construir()
        t0 = time.perf_counter()
        consulta.collect()
        tiempos.append(round(time.perf_counter() - t0, 3))
    medidas = tiempos[1:]
    return {"calentamiento_s": tiempos[0], "medidas_s": medidas, "mediana_s": round(statistics.median(medidas), 3),
            "min_s": min(medidas), "max_s": max(medidas)}


medidas_csv = {v: medir(lambda v=v: consulta_clase(v)) for v in VARIANTES_N1}
print(pd.DataFrame(medidas_csv).T[["calentamiento_s", "medidas_s", "mediana_s", "min_s", "max_s"]].to_string())

           calentamiento_s              medidas_s mediana_s  min_s  max_s
nativa               0.521  [0.588, 0.447, 0.443]     0.447  0.443  0.588
udf_fila             0.783  [0.746, 0.822, 0.725]     0.746  0.725  0.822
udf_pandas           0.595  [0.665, 0.596, 0.649]     0.649  0.596  0.665


In [24]:
# E.2 (continuacion) La trampa: construir la consulta UNA vez y ejecutarla cuatro veces
consulta_unica = consulta_clase("udf_fila")
tiempos_trampa = []
for _ in range(4):
    t0 = time.perf_counter()
    consulta_unica.collect()
    tiempos_trampa.append(round(time.perf_counter() - t0, 3))
print("Misma consulta ejecutada 4 veces (s):", tiempos_trampa)
print("Construida de nuevo en cada corrida (s): calentamiento", medidas_csv["udf_fila"]["calentamiento_s"], "· medidas", medidas_csv["udf_fila"]["medidas_s"])

Misma consulta ejecutada 4 veces (s): [0.805, 0.018, 0.017, 0.016]
Construida de nuevo en cada corrida (s): calentamiento 0.783 · medidas [0.746, 0.822, 0.725]


In [25]:
# E.3 explain(): el nodo del plan que evalua la funcion en Python. Los numeros tras # cambian en cada ejecucion
def lineas_python_del_plan(consulta):
    texto = io.StringIO()
    with contextlib.redirect_stdout(texto):
        consulta.explain()
    return [re.sub(r"#\d+", "#n", linea.strip(" +-:")) for linea in texto.getvalue().splitlines() if "Python" in linea]


nodos_plan = {v: R.nodo_del_plan(consulta_clase(v)) for v in VARIANTES_N1}
for v in VARIANTES_N1:
    print(f"{v}: nodo {nodos_plan[v]}", lineas_python_del_plan(consulta_clase(v)))

# El mismo plan cuando la accion NO usa la clase: un count() sobre filas, con la columna calculada pero sin consumirla
sin_consumir = {v: R.nodo_del_plan(base_variantes.withColumn("nivel_presion", VARIANTES_N1[v]()).groupBy().count()) for v in VARIANTES_N1}
print("Nodo con un count() que no consume la clase:", sin_consumir)

nativa: nodo ninguno []
udf_fila: nodo BatchEvalPython ['BatchEvalPython [clasificar_udf_fila(presion_bar#n)#n], [pythonUDF0#n]']
udf_pandas: nodo ArrowEvalPython ['ArrowEvalPython [clasificar_udf_pandas(presion_bar#n)#n], [pythonUDF0#n], 200']
Nodo con un count() que no consume la clase: {'nativa': 'ninguno', 'udf_fila': 'ninguno', 'udf_pandas': 'ninguno'}


In [26]:
# E.4 Cuantas veces se invoca la funcion, cuantas filas viajan a Python y en cuantos lotes de Arrow: acumuladores dentro de las funciones
from pyspark.accumulators import AccumulatorParam


class ContadorPorTamano(AccumulatorParam):
    """Acumulador de un diccionario {tamano del lote: cuantos lotes}: suma lo que cuenta cada proceso de Python."""
    def zero(self, valor):
        return {}

    def addInPlace(self, a, b):
        for tamano, n in b.items():
            a[tamano] = a.get(tamano, 0) + n
        return a




def variante_contada(nombre):
    """Crea la funcion propia de la variante con acumuladores que cuentan cada invocacion y cada lote."""
    invocaciones = spark.sparkContext.accumulator(0)
    lotes = spark.sparkContext.accumulator({}, ContadorPorTamano())
    if nombre == "udf_fila":
        @F.udf(returnType=StringType())
        def funcion(presion):
            invocaciones.add(1)
            if presion is None:
                return None
            return "baja" if presion < UMBRAL_BAJA else ("alta" if presion > UMBRAL_ALTA else "normal")
    else:
        @F.pandas_udf(StringType())
        def funcion(presion: pd.Series) -> pd.Series:
            invocaciones.add(1)
            lotes.add({len(presion): 1})
            clase = pd.Series("normal", index=presion.index, dtype="object")
            clase[presion < UMBRAL_BAJA] = "baja"
            clase[presion > UMBRAL_ALTA] = "alta"
            clase[presion.isna()] = None
            return clase
    return funcion, invocaciones, lotes


filas_por_particion = [r["count"] for r in crudo.groupBy(F.spark_partition_id().alias("p")).count().orderBy("p").collect()]
conteo_funciones = []
for nombre in ("udf_fila", "udf_pandas"):
    funcion, invocaciones, lotes = variante_contada(nombre)
    clases = base_variantes.withColumn("nivel_presion", funcion(F.col("presion_bar"))).groupBy("nivel_presion").count().collect()
    filas_a_python = invocaciones.value if nombre == "udf_fila" else sum(k * v for k, v in lotes.value.items())
    conteo_funciones.append({"variante": nombre, "invocaciones": invocaciones.value, "filas_serializadas": filas_a_python,
                             "lotes_de_arrow": sum(lotes.value.values()), "tamanos_de_lote": dict(sorted(lotes.value.items(), reverse=True)),
                             "clases": {str(r["nivel_presion"]): r["count"] for r in clases}})
conteo_funciones.insert(0, {"variante": "nativa", "invocaciones": 0, "filas_serializadas": 0, "lotes_de_arrow": 0, "tamanos_de_lote": {}, "clases": conteos["nativa"]})
tabla_invocaciones = pd.DataFrame(conteo_funciones).set_index("variante")
print(tabla_invocaciones.drop(columns="clases").to_string())

lotes_esperados = sum(math.ceil(n / 10_000) for n in filas_por_particion)
print(f"Filas por particion: {filas_por_particion} · lotes esperados = suma de techo(filas / 10 000) = {lotes_esperados}")
print("Lotes de envio de la funcion por fila (100 filas por lote):", math.ceil(576_000 / 100), "· invocaciones por lote de envio:", 100)

            invocaciones  filas_serializadas  lotes_de_arrow       tamanos_de_lote
variante                                                                          
nativa                 0                   0               0                    {}
udf_fila          576000              576000               0                    {}
udf_pandas            60              576000              60  {10000: 56, 4000: 4}
Filas por particion: [144000, 144000, 144000, 144000] · lotes esperados = suma de techo(filas / 10 000) = 60
Lotes de envio de la funcion por fila (100 filas por lote): 5760 · invocaciones por lote de envio: 100


In [27]:
# E.5 Segunda tabla: la columna de presion YA en memoria, para ver el costo de la funcion sin el de leer el CSV
presion_en_memoria = crudo.select("presion_bar").cache()
filas_en_memoria = presion_en_memoria.count()                      # materializa la columna en memoria
almacenada = api("storage/rdd")[0]
medidas_memoria = {v: medir(lambda v=v: consulta_clase(v, presion_en_memoria)) for v in VARIANTES_N1}
presion_en_memoria.unpersist()
print(f"En memoria: {almacenada['numCachedPartitions']} particiones · {almacenada['memoryUsed'] / 2**20:.1f} MiB")
print(pd.DataFrame(medidas_memoria).T[["calentamiento_s", "medidas_s", "mediana_s", "min_s", "max_s"]].to_string())


resumen_csv, resumen_memoria = tabla_resumen(medidas_csv), tabla_resumen(medidas_memoria)
print("\nSobre el CSV (la lectura esta incluida):")
print(resumen_csv.to_string())
print("\nCon la columna en memoria (solo cuesta la clasificacion):")
print(resumen_memoria.to_string())

En memoria: 4 particiones · 4.4 MiB
           calentamiento_s              medidas_s mediana_s  min_s  max_s
nativa               0.222   [0.079, 0.105, 0.06]     0.079   0.06  0.105
udf_fila             0.488  [0.351, 0.344, 0.371]     0.351  0.344  0.371
udf_pandas             0.2   [0.201, 0.209, 0.21]     0.209  0.201   0.21

Sobre el CSV (la lectura esta incluida):
            mediana_s        rango_s  razon_vs_nativa rango_se_solapa_con_nativa
variante                                                                        
nativa          0.447  0.443 a 0.588              1.0                       None
udf_fila        0.746  0.725 a 0.822              1.7                      False
udf_pandas      0.649  0.596 a 0.665              1.5                      False

Con la columna en memoria (solo cuesta la clasificacion):
            mediana_s        rango_s  razon_vs_nativa rango_se_solapa_con_nativa
variante                                                                        


In [28]:
# Autoverificacion del sub-paso E: el comportamiento manda; la tabla de tiempos es la del equipo
assert conteos["nativa"] == conteos["udf_fila"] == conteos["udf_pandas"], "las tres variantes dan el mismo conteo por clase"
assert sum(conteos["nativa"].values()) == 576_000 and set(conteos["nativa"]) == {"baja", "normal", "alta"}
assert all(len(m["medidas_s"]) == 3 for m in medidas_csv.values()), "protocolo: 3 corridas medidas ademas del calentamiento"
assert nodos_plan == {"nativa": "ninguno", "udf_fila": "BatchEvalPython", "udf_pandas": "ArrowEvalPython"}
assert set(sin_consumir.values()) == {"ninguno"}, "un count() que no consume la clase poda la funcion: no aparece ningun nodo"
assert max(tiempos_trampa[1:]) < min(medidas_csv["udf_fila"]["medidas_s"]), "la consulta reutilizada hace casi nada de trabajo en las corridas 1 a 3"
assert tabla_invocaciones.loc["udf_fila", "invocaciones"] == 576_000 and tabla_invocaciones.loc["udf_fila", "filas_serializadas"] == 576_000
assert tabla_invocaciones.loc["udf_pandas", "filas_serializadas"] == 576_000
assert tabla_invocaciones.loc["udf_pandas", "invocaciones"] == tabla_invocaciones.loc["udf_pandas", "lotes_de_arrow"] == lotes_esperados
assert tabla_invocaciones["clases"].apply(lambda c: c == conteos["nativa"]).all()
assert resumen_memoria.loc["udf_fila", "razon_vs_nativa"] > 1, "con la columna en memoria la funcion por fila cuesta mas que la nativa"
print("Autoverificacion del sub-paso E superada: mismo resultado, nodos del plan, invocaciones frente a lotes y protocolo de medicion.")

Autoverificacion del sub-paso E superada: mismo resultado, nodos del plan, invocaciones frente a lotes y protocolo de medicion.


**Su decisión · Sub-paso E.** Adoptamos la **expresión nativa**. Da el mismo conteo por clase que las funciones propias (27 563 / 520 965 / 27 472 en la base) y su plan no tiene nodo de Python. Sobre el CSV de la base, la nativa tuvo 0,45 s de mediana (0,44 a 0,59), la vectorizada 0,65 s (0,60 a 0,67) y la función por fila 0,75 s (0,73 a 0,82). En esta ejecución los rangos no se solapan con la nativa, pero en una ejecución anterior del mismo cuaderno sí se solaparon (nativa 0,60 a 1,20 s): sobre el CSV, la diferencia no es estable, porque la lectura domina. Con la columna en memoria la diferencia sí es estable: la función por fila tardó **4,4 veces** lo que la nativa y la vectorizada **2,6 veces**, sin solape en ninguna de las dos ejecuciones. **No hay diferencia que declarar** cuando los rangos se solapan, o cuando el orden de dos variantes cambia de una ejecución a otra. Una función propia solo se justificaría si la regla no se pudiera escribir con `when`/`otherwise`, y en ese caso usaríamos la vectorizada.

### Tabla de mediciones del Nivel 1

La celda siguiente reúne lo medido en una tabla y la guarda como `mediciones_nivel1.csv` en la carpeta del paquete. Súbanla al repositorio del equipo.

In [29]:
# Tabla de mediciones del Nivel 1: lo que midieron, listo para el repositorio del equipo
mediciones = pd.DataFrame([
    ("Trabajos al leer la cruda: inferida, sin inferir, declarada", f"{f_inferida['trabajos_al_leer']} · {f_texto['trabajos_al_leer']} · {f_declarada['trabajos_al_leer']}", "Jobs"),
    ("MiB leidos en total (lectura y count()): inferida y declarada", f"{f_inferida['mib_total']} · {f_declarada['mib_total']}", "Stages (Input Size)"),
    ("Nulos en consumo_m3 con el tipo entero (PERMISSIVE)", calidad_entero["nulos"]["consumo_m3"], "nulos por columna"),
    ("Filas con el encabezado no declarado", filas_sin_encabezado, "count()"),
    ("Consumo del sector S-ESTE el 2024-01-05: consolidada/ y orden distinto con enforceSchema=True", f"{dia_ref['S-ESTE']} · {con_omision['S-ESTE']}", "suma por sector"),
    ("Filas de la capa nueva y es_confiable verdadero", f"{calidad_capa['filas']} · {confiables}", "resumen de calidad"),
    ("Archivos de la capa, MB y escritura (s)", f"{len(archivos_capa)} · {peso_mb} · {tiempo_escritura}", "arbol de carpetas"),
    ("Tipo fisico y logico de la marca temporal (nueva)", f"{tipos_marca.loc['fisico', 'nueva (refinada_spark/)']} · {tipos_marca.loc['logico', 'nueva (refinada_spark/)']}", "pyarrow"),
    ("Grupos iguales a la heredada y a consolidada/", f"{len(firma_nueva)} · {len(R.referencia_consolidada(LAGO))}", "firma por grupo"),
    ("Hash del contenido de la capa", hash_capa[:16], "firma por grupo"),
    ("Invocaciones de la funcion: por fila y vectorizada", f"{tabla_invocaciones.loc['udf_fila', 'invocaciones']} · {tabla_invocaciones.loc['udf_pandas', 'invocaciones']}", "acumuladores"),
    ("Mediana sobre el CSV (s): nativa, por fila, vectorizada", " · ".join(str(medidas_csv[v]["mediana_s"]) for v in VARIANTES_N1), "protocolo 1 + 3"),
    ("Mediana con la columna en memoria (s): nativa, por fila, vectorizada", " · ".join(str(medidas_memoria[v]["mediana_s"]) for v in VARIANTES_N1), "protocolo 1 + 3"),
], columns=["medicion", "valor", "donde_se_lee"])
mediciones.to_csv(RAIZ / "mediciones_nivel1.csv", index=False, encoding="utf-8")
print(mediciones.to_string(index=False))

                                                                                     medicion                                             valor        donde_se_lee
                                  Trabajos al leer la cruda: inferida, sin inferir, declarada                                         3 · 2 · 0                Jobs
                                MiB leidos en total (lectura y count()): inferida y declarada                                     56.54 · 28.24 Stages (Input Size)
                                          Nulos en consumo_m3 con el tipo entero (PERMISSIVE)                                            576000   nulos por columna
                                                         Filas con el encabezado no declarado                                            576120             count()
Consumo del sector S-ESTE el 2024-01-05: consolidada/ y orden distinto con enforceSchema=True                                    190.7 · 3601.5     suma por sector
                

In [30]:
# Cierre del Nivel 1: el datalake/ heredado no cambio (la capa nueva se escribe en refinada_spark/, que no se cuenta)
datalake_despues, archivos_despues = hash_arbol(LAGO)
assert (datalake_despues, archivos_despues) == (datalake_antes, archivos_antes), "datalake/ debe seguir identico"
print(f"datalake/ identico: {archivos_despues} archivos · SHA-256 {datalake_despues[:16]}...")

datalake/ identico: 360 archivos · SHA-256 0a986ffec04ad666...


**Consolidación (por equipo).**

(1) Declarar el esquema evitó el error de **`consumo_m3` como entero**, que deja 576 000 nulos y una reconciliación con 0 de 480 grupos, y el de las **columnas reordenadas**, que deja consumos 19 veces mayores. Sin esquema declarado, solo una reconciliación de sumas por grupo contra una referencia independiente habría detectado los dos errores. `count()` no habría detectado ninguno.

(2) Si el volumen del cierre se multiplicara por 10, lo primero que cambiaría son las **invocaciones de la función por fila**, que crecen 1 a 1 con las filas: de 576 000 a 5 760 000. También cambiaría su tiempo con la columna en memoria. La nativa y los lotes de Arrow crecen mucho menos.

(3) Para la directora financiera: *"Clasificar la presión con la herramienta que trae el motor es igual de exacto y, en nuestras mediciones, entre 4 y 11 veces más rápido en el paso de clasificar que hacerlo con un programa propio. Un programa propio solo vale la pena si la regla no se puede escribir con la herramienta del motor."*

## Nivel 2 · Aplicado · 1,5 horas del presupuesto de la tarea

> **Antes de empezar el Nivel 2: qué viene del Nivel 1.** Las celdas del Nivel 2 reutilizan lo que definió el Nivel 1 en el mismo núcleo: `ESQUEMA` (celda C.1), `MODOS` y `filas_y_suma` (celda C.2), los ayudantes (`tipo_marca_temporal`, `tabla_resumen`) y la configuración común. Si reinician el núcleo (por ejemplo, porque `docker stats` se acercó a los 2,9 GiB), ejecuten de nuevo, en este orden: la configuración común, la celda de ayudantes y las celdas C.1 y C.2, y vuelvan a abrir la sesión con `spark = abrir_sesion()` (la función `abrir_sesion` se define en la configuración común; la celda B.2 solo la invoca). La capa de la base del sub-paso D sigue en disco.

**Presupuesto (estimado).** Estas 1,5 horas (90 minutos) se reparten así: generar la ampliación y abrir la sesión, 5 min; completar `definir_esquema`, `columnas_derivadas` y `clasificar_nativa`, con los modos de fallo que no alcanzaron en clase, 25 min; escribir `escribir_capa`, dos corridas por volumen y la reconciliación, 15 min; reproceso de 6 días, 10 min; comparación de variantes sobre la ampliación (12 corridas por tabla), 15 min; cierre diario con DataFrame frente al RDD, 5 min; lectura con pyarrow, bitácora y evidencia, 12 min; colchón, 3 min. El resto de las 4 horas de la tarea es para el registro de decisiones, la nota para la gerencia, el puente de modelado, el índice de evidencias y la lectura en inglés. Los tiempos de reloj varían hasta ×3 según la carga del equipo: la ejecución ocupa unos 12 minutos en total; lo que consume el tiempo es escribir y redactar.

### El encargo

Repitan el patrón del Nivel 1 sobre **la ampliación de su equipo**, con el código de la capa en `pipeline/refinada_spark.py`. Cambien `EQUIPO` en la celda de configuración (o definan la variable de entorno `EQUIPO`) y ejecuten en orden.

**Paso mecánico (N2.1).** La celda genera la ampliación en `datalake_equipo/`, comprueba que `datalake/` no cambió (el hash antes y después debe ser el mismo) y muestra cómo se reparte **su** cruda entre las particiones del lector.

In [31]:
# N2.1 Generar la ampliacion del equipo y comprobar que datalake/ NO cambio
EQUIPO_CARPETA = RAIZ / "datalake_equipo"
EQUIPO_CRUDA = EQUIPO_CARPETA / "cruda" / "acueducto"

antes, archivos_antes_gen = hash_arbol(LAGO)
t0 = time.time()
salida_generador = subprocess.run([sys.executable, str(RAIZ / "practica" / "generar_ampliacion_s15.py"), "--equipo", EQUIPO],
                                  cwd=RAIZ, capture_output=True, text=True)
tiempo_generacion = round(time.time() - t0, 1)
print(salida_generador.stdout or salida_generador.stderr)
despues, archivos_despues_gen = hash_arbol(LAGO)
assert salida_generador.returncode == 0, "el generador fallo: revisen el identificador del equipo"
assert (antes, archivos_antes_gen) == (despues, archivos_despues_gen), "el generador no debe tocar datalake/"

perfil = json.loads((EQUIPO_CARPETA / "parametros_equipo.json").read_text(encoding="utf-8"))
disco_mib = round(sum(f.stat().st_size for f in EQUIPO_CARPETA.rglob("*") if f.is_file() and "refinada_spark" not in f.parts and "consolidada_spark" not in f.parts) / 2**20, 1)
print({k: perfil[k] for k in ("equipo", "medidores", "lecturas_totales", "sector_dominante", "fraccion_dominante")})
print(f"Disco ocupado: {disco_mib} MiB · tiempo de generacion: {tiempo_generacion} s · datalake/ identico ({archivos_despues_gen} archivos)")

Equipo equipo-1: 348 medidores, 120 dias, 1,002,240 lecturas
Sector dominante S-SUR: 120 medidores (34.5%); cada uno de los otros tres: 76
Escrito en datalake_equipo (56.0 MiB, 3.2 s)



{'equipo': 'equipo-1', 'medidores': 348, 'lecturas_totales': 1002240, 'sector_dominante': 'S-SUR', 'fraccion_dominante': 0.3448}
Disco ocupado: 56.0 MiB · tiempo de generacion: 3.5 s · datalake/ identico (360 archivos)


In [32]:
# N2.1 (continuacion) Como se reparte SU cruda entre las particiones del lector: de ese reparto dependen los lotes de Arrow
spark = abrir_sesion()
crudo_equipo = R.leer_cruda(spark, EQUIPO_CRUDA, ESQUEMA)
particion = F.spark_partition_id().alias("particion")
filas_equipo = [r["count"] for r in crudo_equipo.groupBy(particion).count().orderBy("particion").collect()]
archivos_equipo = [r["count"] for r in crudo_equipo.select(particion, F.input_file_name().alias("archivo")).distinct()
                   .groupBy("particion").count().orderBy("particion").collect()]
lotes_equipo = sum(math.ceil(n / 10_000) for n in filas_equipo)
print("Particiones del lector:", len(filas_equipo), "· archivos por particion:", archivos_equipo)
print("Filas por particion:", filas_equipo, "· total:", sum(filas_equipo))
print("Lotes de Arrow esperados para la funcion vectorizada (suma de techo(filas / 10 000)):", lotes_equipo)

Particiones del lector: 5 · archivos por particion: [29, 29, 29, 29, 4]
Filas por particion: [242208, 242208, 242208, 242208, 33408] · total: 1002240
Lotes de Arrow esperados para la funcion vectorizada (suma de techo(filas / 10 000)): 104


### N2.2 Su turno: completar tres funciones del script

`definir_esquema`, `columnas_derivadas` y `clasificar_nativa` son tres de las cuatro funciones que faltan en `pipeline/refinada_spark.py`; la cuarta, `escribir_capa`, está en N2.4. La sesión, la lectura, las dos funciones propias, la firma por grupo, la reconciliación y la bitácora ya funcionan. Cada celda de ejercicio va seguida de su autoverificación sobre la base, que reproduce lo que ya hicieron en el Nivel 1. Cuando las cuatro pasen, **llévenlas a `pipeline/refinada_spark.py`** (reemplacen cada `NotImplementedError`) y ejecuten la celda de N2.4b.

**N2.2a · `definir_esquema`**

In [33]:
def definir_esquema():
    """Esquema DECLARADO de la capa cruda.

    TODO (equipo): devuelvan un StructType con las seis columnas del CSV, en el MISMO orden
    del encabezado: meter_id, timestamp, sector, consumo_m3, presion_bar y quality_flag.
      - Elijan el tipo de cada una (texto, marca temporal, numero) y justifiquenlo en su
        registro de decisiones: que pasa con un tipo entero para consumo_m3? y con un
        float en lugar de un double?
      - Con esquema declarado los tipos se asignan por POSICION, no por nombre.
      - Ninguna columna se infiere.

    Decision (equipo-1): meter_id, sector y quality_flag como texto (son codigos, no numeros:
    'MED0001' o 'S-SUR' no admiten aritmetica); timestamp como TimestampType (instante en UTC
    con precision de microsegundos, la de la capa); consumo_m3 y presion_bar como DoubleType.
    Un entero truncaria 0,209 m3 a 0 y el consumo diario se perderia casi entero; un float de
    32 bits tiene ~7 cifras significativas y la suma de 1 200 lecturas por grupo acumularia un
    error del orden de 1e-4 m3, mayor que la tolerancia de 1e-6 de la reconciliacion."""
    return StructType([
        StructField("meter_id", StringType(), True),
        StructField("timestamp", TimestampType(), True),
        StructField("sector", StringType(), True),
        StructField("consumo_m3", DoubleType(), True),
        StructField("presion_bar", DoubleType(), True),
        StructField("quality_flag", StringType(), True),
    ])


In [34]:
# Autoverificacion de definir_esquema: el contrato se prueba leyendo la BASE con el
esquema_equipo = definir_esquema()
assert isinstance(esquema_equipo, StructType), "definir_esquema debe devolver un StructType"
assert esquema_equipo.names == ["meter_id", "timestamp", "sector", "consumo_m3", "presion_bar", "quality_flag"], "las seis columnas, en el orden del encabezado"
tipos_equipo = {f.name: f.dataType for f in esquema_equipo.fields}
assert isinstance(tipos_equipo["timestamp"], TimestampType), "timestamp se declara como marca temporal"
assert isinstance(tipos_equipo["consumo_m3"], DoubleType) and isinstance(tipos_equipo["presion_bar"], DoubleType), "consumo_m3 y presion_bar son double"
assert all(isinstance(tipos_equipo[c], StringType) for c in ("meter_id", "sector", "quality_flag")), "meter_id, sector y quality_flag son texto"

spark = abrir_sesion()
lectura_equipo = R.leer_cruda(spark, CRUDA, esquema_equipo)
calidad_equipo = R.resumen_calidad(lectura_equipo)
assert calidad_equipo["filas"] == 576_000, "la lectura de la base debe dar 576 000 filas"
assert sum(calidad_equipo["nulos"].values()) == 0, f"un tipo mal declarado dejo nulos sin error: {calidad_equipo['nulos']}"
print("definir_esquema correcta: 576 000 filas y 0 nulos en las seis columnas")

definir_esquema correcta: 576 000 filas y 0 nulos en las seis columnas


**N2.2b · `columnas_derivadas`**

In [35]:
def columnas_derivadas(df):
    """Deriva las columnas de la capa desde la marca temporal y la bandera de calidad.

    TODO (equipo): agreguen estas columnas con withColumn, todas a partir de las existentes:
      - fecha: la fecha de la lectura (tipo fecha).
      - hora: la hora del dia, de 0 a 23.
      - es_confiable: verdadero si quality_flag es 'OK'. La capa NO filtra filas.
      - anio, mes y dia: las tres partes de la ruta como TEXTO con ceros ('2024', '01',
        '05'), tomadas de la marca temporal. La convencion del lago es anio=AAAA/mes=MM/dia=DD:
        que ocurre si mes queda como entero?
    Devuelvan el DataFrame con las columnas nuevas; nivel_presion se agrega despues.

    Si mes quedara entero, la ruta seria mes=1 en lugar de mes=01: rompe la convencion del lago,
    el orden lexicografico de las carpetas (mes=10 antes de mes=2) y cualquier lector que
    construya la ruta con ceros."""
    ts = F.col("timestamp")
    return (df.withColumn("fecha", F.to_date(ts))
              .withColumn("hora", F.hour(ts))
              .withColumn("es_confiable", F.col("quality_flag") == F.lit("OK"))
              .withColumn("anio", F.date_format(ts, "yyyy"))
              .withColumn("mes", F.date_format(ts, "MM"))
              .withColumn("dia", F.date_format(ts, "dd")))


In [36]:
# Autoverificacion de columnas_derivadas sobre la BASE
spark = abrir_sesion()
lectura = R.leer_cruda(spark, CRUDA, definir_esquema())
derivada = columnas_derivadas(lectura)
tipos_derivada = dict(derivada.dtypes)
for nombre in ("fecha", "hora", "es_confiable", "anio", "mes", "dia"):
    assert nombre in tipos_derivada, f"falta la columna {nombre}"
assert (tipos_derivada["fecha"], tipos_derivada["hora"], tipos_derivada["es_confiable"]) == ("date", "int", "boolean"), "tipos de fecha, hora y es_confiable"
assert tipos_derivada["anio"] == tipos_derivada["mes"] == tipos_derivada["dia"] == "string", "anio, mes y dia deben ser TEXTO con ceros, no enteros"
resumen_derivada = derivada.agg(F.count("*").alias("filas"), F.sum(F.col("es_confiable").cast("int")).alias("confiables"),
                                F.collect_set("anio").alias("anios"), F.collect_set("mes").alias("meses"),
                                F.min("dia").alias("dia_min"), F.max("dia").alias("dia_max"),
                                F.min("hora").alias("hora_min"), F.max("hora").alias("hora_max")).collect()[0]
assert resumen_derivada["filas"] == 576_000, "la capa conserva todas las filas: no se filtra por calidad"
assert resumen_derivada["confiables"] == 541_368, "es_confiable es verdadero en 541 368 filas"
assert sorted(resumen_derivada["anios"]) == ["2024"] and sorted(resumen_derivada["meses"]) == ["01", "02", "03", "04"], "mes con cero a la izquierda"
assert (resumen_derivada["dia_min"], resumen_derivada["dia_max"]) == ("01", "31") and (resumen_derivada["hora_min"], resumen_derivada["hora_max"]) == (0, 23)
print("columnas_derivadas correcta: 576 000 filas, 541 368 confiables, mes como '01' a '04', hora de 0 a 23")

columnas_derivadas correcta: 576 000 filas, 541 368 confiables, mes como '01' a '04', hora de 0 a 23


**N2.2c · `clasificar_nativa`**

In [37]:
def clasificar_nativa(presion):
    """Expresion nativa de columna para clasificar la presion.

    TODO (equipo): reciban una COLUMNA (presion) y devuelvan una columna con 'baja' si la
    presion es menor que UMBRAL_BAJA, 'alta' si es mayor que UMBRAL_ALTA y 'normal' en otro
    caso. Usen when(...).when(...) de pyspark.sql.functions, sin funciones de Python.
    Decidan que clase le corresponde a una presion nula y que sea la misma que dan las
    dos funciones propias de abajo: las tres variantes deben producir el mismo conteo por clase.

    Decision (equipo-1): una presion nula queda con clase NULA, igual que en las dos funciones
    propias (None). Inventarle 'normal' ocultaria un dato faltante como si fuera sano."""
    return (F.when(presion < UMBRAL_BAJA, "baja")
             .when(presion > UMBRAL_ALTA, "alta")
             .when(presion.isNotNull(), "normal"))


In [38]:
# Autoverificacion de clasificar_nativa: fronteras, nulo y conteo por clase de la BASE
spark = abrir_sesion()
prueba = spark.createDataFrame([(1.0,), (2.4,), (2.5,), (3.0,), (3.5,), (3.6,), (None,)], "presion_bar double")
clases_prueba = [r["nivel_presion"] for r in prueba.withColumn("nivel_presion", clasificar_nativa(F.col("presion_bar"))).collect()]
assert clases_prueba == ["baja", "baja", "normal", "normal", "normal", "alta", None], f"fronteras y nulo: {clases_prueba}"
lectura = R.leer_cruda(spark, CRUDA, definir_esquema())
conteo_nativa = {str(r["nivel_presion"]): r["count"] for r in
                 lectura.withColumn("nivel_presion", clasificar_nativa(F.col("presion_bar"))).groupBy("nivel_presion").count().collect()}
assert conteo_nativa == {"baja": 27_563, "normal": 520_965, "alta": 27_472}, f"conteo por clase: {conteo_nativa}"
print("clasificar_nativa correcta: 2,5 y 3,5 son normales, la presion nula da clase nula y el conteo es", conteo_nativa)

clasificar_nativa correcta: 2,5 y 3,5 son normales, la presion nula da clase nula y el conteo es {'baja': 27563, 'normal': 520965, 'alta': 27472}


### N2.3 Modos de fallo que no alcanzaron en clase

Tres celdas ya escritas (menos de 1 minuto de ejecución en total): el formato de fecha errado y `nullable=False` (lectura), el tipo de retorno equivocado y el `None` que llega a la función (funciones propias) y la colisión con las columnas de la ruta (releer la capa). Ejecútenlas mientras redactan las funciones y anoten en su registro de decisiones qué prueba detectó cada una.

In [39]:
# N2.3 Modos de fallo de la lectura que no alcanzaron en clase: formato de fecha errado y nullable=False
spark = abrir_sesion()
formato_errado = {modo: spark.read.csv(str(CRUDA), header=True, schema=ESQUEMA, timestampFormat="dd/MM/yyyy HH:mm:ss", mode=modo) for modo in MODOS}


def filas_suma_y_marcas(df):
    """En la MISMA agregacion: filas, suma de consumo y marcas temporales no nulas (obliga a convertir timestamp)."""
    fila = df.agg(F.count("*").alias("filas"), F.sum("consumo_m3").alias("suma"), F.count("timestamp").alias("marcas")).collect()[0]
    return fila["filas"], round(fila["suma"], 1) if fila["suma"] is not None else None, fila["marcas"]


def nulos_de_timestamp(df):
    return df.agg(F.sum(F.col("timestamp").isNull().cast("int"))).collect()[0][0]


tabla_fecha = pd.DataFrame([{"modo": modo, "count()": intentar(df.count), "nulos en timestamp": intentar(lambda df=df: nulos_de_timestamp(df)),
                             "filas, suma de consumo y marcas no nulas": intentar(lambda df=df: filas_suma_y_marcas(df))}
                            for modo, df in formato_errado.items()])
print("Formato de fecha 'dd/MM/yyyy HH:mm:ss' sobre un texto 'AAAA-MM-DD HH:MM:SS':")
print(tabla_fecha.to_string(index=False))

esquema_sin_nulos = StructType([StructField(f.name, f.dataType, False) for f in ESQUEMA.fields])
lectura_sin_nulos = spark.read.csv(str(CRUDA), header=True, schema=esquema_sin_nulos)
entero_sin_nulos = spark.read.csv(str(CRUDA), header=True, schema=StructType([StructField(f.name, IntegerType() if f.name == "consumo_m3" else f.dataType, False) for f in ESQUEMA.fields]))
print("\nnullable=False declarado, nullable leido:", {f.name: f.nullable for f in lectura_sin_nulos.schema.fields if f.name in ESQUEMA.names})
print("Con consumo_m3 entero y nullable=False, nulos en consumo_m3:", R.resumen_calidad(entero_sin_nulos)["nulos"]["consumo_m3"], "(sin ningun error)")

26/11/12 22:05:56 ERROR Executor: Exception in task 3.0 in stage 240.0 (TID 599)
org.apache.spark.SparkException: Encountered error while reading file file:///proyecto/datalake/cruda/acueducto/anio=2024/mes=02/dia=25/lecturas.csv. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:864)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:293)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithoutKey_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.e

26/11/12 22:05:57 ERROR Executor: Exception in task 2.0 in stage 241.0 (TID 602)
org.apache.spark.SparkException: Encountered error while reading file file:///proyecto/datalake/cruda/acueducto/anio=2024/mes=04/dia=13/lecturas.csv. Details:
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:864)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:293)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithoutKey_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.e

Formato de fecha 'dd/MM/yyyy HH:mm:ss' sobre un texto 'AAAA-MM-DD HH:MM:SS':
         modo  count()                 nulos en timestamp filas, suma de consumo y marcas no nulas
   PERMISSIVE   576000                             576000                     (576000, 92166.0, 0)
DROPMALFORMED   576000                               None                             (0, None, 0)
     FAILFAST   576000 ERROR: MALFORMED_RECORD_IN_PARSING       ERROR: MALFORMED_RECORD_IN_PARSING

nullable=False declarado, nullable leido: {'meter_id': True, 'timestamp': True, 'sector': True, 'consumo_m3': True, 'presion_bar': True, 'quality_flag': True}


Con consumo_m3 entero y nullable=False, nulos en consumo_m3: 576000 (sin ningun error)


In [40]:
# N2.3 Modos de fallo de las funciones propias: tipo de retorno equivocado y un None que llega a la funcion (sobre un dia: 4 800 filas)
spark = abrir_sesion()
un_dia = R.leer_cruda(spark, CRUDA / "anio=2024" / "mes=01" / "dia=01", ESQUEMA)


@F.udf(returnType=IntegerType())
def fila_retorno_equivocado(presion):
    return "baja" if presion < 2.5 else "normal"       # devuelve texto pero declara entero


@F.pandas_udf(IntegerType())
def pandas_retorno_equivocado(presion: pd.Series) -> pd.Series:
    return pd.Series(["normal"] * len(presion), index=presion.index)       # texto donde se declaro entero


@F.udf(returnType=StringType())
def fila_sin_guarda(presion):
    return "baja" if presion < 2.5 else ("alta" if presion > 3.5 else "normal")       # no contempla None


con_presion_nula = un_dia.withColumn("presion_bar", F.when(F.col("meter_id") == "MED0001", F.lit(None).cast("double")).otherwise(F.col("presion_bar")))
resultado_fila = intentar(lambda: {str(r["c"]): r["count"] for r in un_dia.withColumn("c", fila_retorno_equivocado("presion_bar")).groupBy("c").count().collect()})
resultado_pandas = intentar(lambda: un_dia.withColumn("c", pandas_retorno_equivocado("presion_bar")).groupBy("c").count().collect())
resultado_none = intentar(lambda: con_presion_nula.withColumn("c", fila_sin_guarda("presion_bar")).groupBy("c").count().collect())
# La variante nativa usa SU clasificar_nativa del cuaderno (N2.2c): la del script se completa recien en N2.4
columna_de = {"nativa": lambda: clasificar_nativa(F.col("presion_bar")),
              "udf_fila": lambda: R.clasificar_udf_fila(F.col("presion_bar")),
              "udf_pandas": lambda: R.clasificar_udf_pandas(F.col("presion_bar"))}
tres_variantes_con_nulo = {v: {str(r["c"]): r["count"] for r in con_presion_nula.withColumn("c", columna_de[v]()).groupBy("c").count().collect()} for v in columna_de}
print("UDF por fila con tipo de retorno equivocado:", resultado_fila)
print("Pandas UDF con tipo de retorno equivocado:  ", resultado_pandas)
print("UDF por fila sin guarda para None:          ", resultado_none)
print("Tres variantes con presion nula en un medidor:", tres_variantes_con_nulo)

26/11/12 22:06:00 ERROR Executor: Exception in task 0.0 in stage 248.0 (TID 611)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/sql/pandas/serializers.py", line 241, in _create_array
    return pa.Array.from_pandas(
           ^^^^^^^^^^^^^^^^^^^^^
  File "pyarrow/array.pxi", line 1115, in pyarrow.lib.Array.from_pandas
  File "pyarrow/array.pxi", line 339, in pyarrow.lib.array
  File "pyarrow/array.pxi", line 85, in pyarrow.lib._ndarray_to_array
  File "pyarrow/error.pxi", line 91, in pyarrow.lib.check_status
pyarrow.lib.ArrowInvalid: Could not convert 'normal' with type str: tried to convert to int32

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/usr/local/lib/python3.11

26/11/12 22:06:01 ERROR Executor: Exception in task 0.0 in stage 249.0 (TID 612)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/tmp/ipykernel_16013/770972590.py", line 18, in fila_sin_guarda
TypeError: '<' not supported between instances of 'NoneType' and 'float'

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:491)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	

UDF por fila con tipo de retorno equivocado: {'None': 4800}
Pandas UDF con tipo de retorno equivocado:   ERROR: ArrowInvalid
UDF por fila sin guarda para None:           ERROR: PythonException
Tres variantes con presion nula en un medidor: {'nativa': {'baja': 233, 'None': 24, 'normal': 4316, 'alta': 227}, 'udf_fila': {'baja': 233, 'None': 24, 'normal': 4316, 'alta': 227}, 'udf_pandas': {'baja': 233, 'None': 24, 'normal': 4316, 'alta': 227}}


In [41]:
# N2.3 Colision con las columnas de la ruta: al releer la capa, Spark agrega anio, mes y dia como ENTERO
spark = abrir_sesion()
relectura = spark.read.parquet(str(LAGO / "refinada_spark"))
tipos_ruta = {c: t for c, t in relectura.dtypes if c in ("anio", "mes", "dia")}
primer_mes = relectura.select("mes").limit(1).collect()[0]["mes"]
spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")
relectura_texto = spark.read.parquet(str(LAGO / "refinada_spark"))
primer_mes_texto = relectura_texto.select("mes").limit(1).collect()[0]["mes"]
spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "true")
print("Tipos con la configuracion por omision:", tipos_ruta, "· mes de la primera fila:", repr(primer_mes))
print("Sin inferencia de tipo de las columnas de particion:", dict(relectura_texto.dtypes)["mes"], "· mes de la primera fila:", repr(primer_mes_texto))

Tipos con la configuracion por omision: {'anio': 'int', 'mes': 'int', 'dia': 'int'} · mes de la primera fila: 1
Sin inferencia de tipo de las columnas de particion: string · mes de la primera fila: '01'


### N2.4 Su turno: `escribir_capa`

Aquí están dos decisiones de ingeniería de este nivel: el tipo de la marca temporal y la forma de sobrescribir. La autoverificación escribe tres días en una carpeta temporal, lee con pyarrow el tipo físico y lógico de la marca temporal y reprocesa un día con las dos sobrescrituras.

In [42]:
def escribir_capa(df, ruta, sobrescritura="dinamica"):
    """Escribe la capa en Parquet particionado por dia.

    TODO (equipo):
      - Marca temporal en MICROSEGUNDOS: fijen spark.sql.parquet.outputTimestampType en
        TIMESTAMP_MICROS (el valor por omision es INT96, que otros lectores interpretan distinto).
      - Particion por anio, mes y dia (partitionBy) con modo overwrite.
      - Sobrescritura: fijen spark.sql.sources.partitionOverwriteMode en 'dynamic' o 'static'
        segun el parametro `sobrescritura` ('dinamica' o 'estatica'). Con la estatica, escribir
        una ventana de dias borra toda la tabla. Despues de escribir, guarden en su registro
        cuantas filas quedaron en cada caso.
    Usen df.sparkSession para acceder a la configuracion.

    Decision (equipo-1): un archivo por dia (repartition por anio, mes y dia antes de escribir):
    4 800 filas por dia en la base y de 8 352 a 13 680 en la ampliacion caben en un archivo de
    unos cientos de KiB; varios archivos por carpeta solo multiplicarian metadatos."""
    spark = df.sparkSession
    spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
    spark.conf.set("spark.sql.sources.partitionOverwriteMode",
                   {"dinamica": "dynamic", "estatica": "static"}[sobrescritura])
    (df.repartition("anio", "mes", "dia")
       .write.mode("overwrite")
       .partitionBy("anio", "mes", "dia")
       .parquet(str(ruta)))


In [43]:
# Autoverificacion de escribir_capa con TRES dias: tipo de la marca temporal, rutas con ceros y sobrescritura
spark = abrir_sesion()
tres_dias = [str(CRUDA / "anio=2024" / "mes=01" / f"dia=0{d}") for d in (1, 2, 3)]
capa_prueba = columnas_derivadas(spark.read.csv(tres_dias, header=True, schema=definir_esquema(), enforceSchema=False))
capa_prueba = capa_prueba.withColumn("nivel_presion", clasificar_nativa(F.col("presion_bar")))
carpeta_prueba = Path(tempfile.mkdtemp(prefix="capa_")) / "capa"
escribir_capa(capa_prueba, carpeta_prueba)

dias_escritos = sorted(p.relative_to(carpeta_prueba).as_posix() for p in carpeta_prueba.glob("anio=*/mes=*/dia=*"))
assert dias_escritos == ["anio=2024/mes=01/dia=01", "anio=2024/mes=01/dia=02", "anio=2024/mes=01/dia=03"], f"rutas con ceros: {dias_escritos}"
tipo_escrito = tipo_marca_temporal(sorted(carpeta_prueba.rglob("*.parquet"))[0])
assert tipo_escrito["fisico"] == "INT64" and tipo_escrito["logico"] == "isAdjustedToUTC=true, unidad=microseconds", f"marca temporal: {tipo_escrito}"
assert spark.read.parquet(str(carpeta_prueba)).count() == 14_400, "tres dias de 4 800 lecturas"

un_solo_dia = capa_prueba.filter(F.col("dia") == "02")
for modo, esperadas in (("dinamica", 3), ("estatica", 1)):
    copia_prueba = carpeta_prueba.parent / f"copia_{modo}"
    shutil.copytree(carpeta_prueba, copia_prueba)
    escribir_capa(un_solo_dia, copia_prueba, modo)
    carpetas = len(list(copia_prueba.glob("anio=*/mes=*/dia=*")))
    assert carpetas == esperadas, f"reprocesar un dia con sobrescritura {modo} debe dejar {esperadas} carpetas y dejo {carpetas}"
print("escribir_capa correcta: INT64 en microsegundos, rutas con ceros; reprocesar un dia deja 3 carpetas (dinamica) o 1 (estatica)")

escribir_capa correcta: INT64 en microsegundos, rutas con ceros; reprocesar un dia deja 3 carpetas (dinamica) o 1 (estatica)


**N2.4b · Las cuatro funciones, en el script.** Cuando las cuatro autoverificaciones pasen, lleven las funciones a `pipeline/refinada_spark.py` (explorador de archivos de JupyterLab, carpeta `pipeline/`) y ejecuten:

In [44]:
# Autoverificacion: lleven sus cuatro funciones a pipeline/refinada_spark.py (reemplacen cada NotImplementedError) y recarguen el modulo
importlib.reload(R)
pendientes = [n for n in ("definir_esquema", "columnas_derivadas", "clasificar_nativa", "escribir_capa") if "NotImplementedError" in inspect.getsource(getattr(R, n))]
assert not pendientes, f"siguen sin completar en pipeline/refinada_spark.py: {pendientes}"
print("pipeline/refinada_spark.py esta completo: las cuatro funciones ya no lanzan NotImplementedError")

pipeline/refinada_spark.py esta completo: las cuatro funciones ya no lanzan NotImplementedError


### N2.5 Dos corridas por volumen con el script y reconciliación independiente

`correr_capa` ejecuta la función `main` del script, el equivalente de `python pipeline/refinada_spark.py --equipo ... --volumen ...`. Cada corrida anexa una línea a `bitacora_refinada_spark.jsonl`. Dos corridas sobre el mismo volumen deben dejar **el mismo contenido** (mismo número de filas, mismas sumas y mismo hash); el nombre de los archivos `part-*.parquet` cambia y no cuenta. La ejecución de las cuatro corridas ocupa unos 4 minutos: es buen momento para redactar. La lectura independiente repite la reconciliación **sin Spark**, con pyarrow y pandas, contra `consolidada/` (base) y `totales_control.csv` (ampliación).

In [45]:
# N2.5 Ayudantes: ejecutar el script (su funcion main) y leer lo que anexo a la bitacora
def leer_bitacora(nombre="bitacora_refinada_spark.jsonl"):
    ruta = RAIZ / nombre
    return [json.loads(l) for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip()] if ruta.exists() else []


def correr_capa(volumen, variante="nativa", desde=None, hasta=None, salida=None, sobrescritura="dinamica",
                bitacora="bitacora_refinada_spark.jsonl", comparar=False):
    """Equivale a:  python pipeline/refinada_spark.py --equipo EQUIPO --volumen V [--variante ..] [--desde ..] [--hasta ..] [--salida ..]
    main() cierra la sesion de Spark al terminar: la interfaz de esa corrida desaparece, pero sus cifras quedan en la bitacora.
    Devuelve (codigo de salida, lineas nuevas de la bitacora)."""
    argv = ["--equipo", EQUIPO, "--raiz", str(RAIZ), "--volumen", volumen, "--variante", variante, "--master", "local[4]",
            "--driver-memory", DRIVER_MEMORY, "--sobrescritura", sobrescritura, "--bitacora", bitacora]
    for opcion, valor in (("--desde", desde), ("--hasta", hasta), ("--salida", salida)):
        if valor:
            argv += [opcion, str(valor)]
    if comparar:
        argv.append("--comparar-variantes")
    previas = len(leer_bitacora(bitacora))
    with contextlib.redirect_stdout(io.StringIO()):          # main imprime el registro: aqui se lee de la bitacora
        codigo = R.main(argv)
    return codigo, leer_bitacora(bitacora)[previas:]


def agregado_con_pyarrow(carpeta_capa):
    """Lee la capa con pyarrow y pandas (SIN Spark) y agrega filas y consumo por (fecha, sector), sin filtro de calidad."""
    archivos = sorted(Path(carpeta_capa).glob("anio=*/mes=*/dia=*/*.parquet"))
    t = pd.concat([pq.read_table(a, columns=["sector", "consumo_m3", "fecha"]).to_pandas() for a in archivos], ignore_index=True)
    t["fecha"] = t["fecha"].astype(str)
    return t.groupby(["fecha", "sector"]).agg(n_lecturas=("consumo_m3", "size"), consumo_total_m3=("consumo_m3", "sum")).reset_index()


def diferencia_con_referencia(agregado, referencia):
    """Compara el agregado de la capa con una referencia (consolidada/ o totales_control.csv)."""
    m = agregado.merge(referencia, on=["fecha", "sector"], how="outer", suffixes=("_capa", "_ref"), indicator=True)
    return {"grupos": len(m), "solo_en_un_lado": int((m["_merge"] != "both").sum()),
            "lecturas_distintas": int((m["n_lecturas_capa"] != m["n_lecturas_ref"]).sum()),
            "diferencia_max_consumo": float((m["consumo_total_m3_capa"] - m["consumo_total_m3_ref"]).abs().max())}

In [46]:
# N2.5 Dos corridas por volumen, SIN filtro de calidad: la salida debe tener el mismo contenido (mismo hash) y reconciliar
corridas = []
for volumen in ("base", "ampliacion"):
    for numero in (1, 2):
        codigo, nuevas = correr_capa(volumen)
        r = nuevas[-1]
        corridas.append({"volumen": volumen, "corrida": numero, "filas": r["filas_escritas"], "archivos": r["archivos_escritos"],
                         "hash": r["hash_salida"][:12], "no_conformes": r["filas_no_conformes"], "heredada": r["reconciliacion_heredada"],
                         "control": r["reconciliacion_control"], "mib_leidos": r["mib_leidos"], "tiempo_s": r["tiempo_s"]})
tabla_corridas = pd.DataFrame(corridas)
print(tabla_corridas.to_string(index=False))

   volumen  corrida   filas  archivos         hash  no_conformes heredada control  mib_leidos  tiempo_s
      base        1  576000       120 9cc455c0aff8             0       OK      OK       56.48     2.986
      base        2  576000       120 9cc455c0aff8             0       OK      OK       56.48     2.930
ampliacion        1 1002240       120 7b0ac06a1ab1             0       OK      OK      103.71     3.717
ampliacion        2 1002240       120 7b0ac06a1ab1             0       OK      OK      103.71     3.657


In [47]:
# N2.5 Lectura independiente: pyarrow y pandas, sin Spark, contra consolidada/ (base) y totales_control.csv (ampliacion)
independiente = {}
for volumen, carpeta_capa, referencia_volumen in (
        ("base", LAGO / "refinada_spark", R.referencia_consolidada(LAGO)),
        ("ampliacion", EQUIPO_CARPETA / "refinada_spark", R.referencia_totales_control(EQUIPO_CARPETA))):
    independiente[volumen] = diferencia_con_referencia(agregado_con_pyarrow(carpeta_capa), referencia_volumen)
    independiente[volumen]["marca_temporal"] = tipo_marca_temporal(sorted(carpeta_capa.rglob("*.parquet"))[0])
tabla_independiente = pd.DataFrame(independiente).T
print(tabla_independiente.drop(columns="marca_temporal").to_string())
print("Marca temporal de la ampliacion:", independiente["ampliacion"]["marca_temporal"])

           grupos solo_en_un_lado lecturas_distintas diferencia_max_consumo
base          480               0                  0                    0.0
ampliacion    480               0                  0                    0.0
Marca temporal de la ampliacion: {'fisico': 'INT64', 'logico': 'isAdjustedToUTC=true, unidad=microseconds', 'tipo_arrow': 'timestamp[us, tz=UTC]'}


In [48]:
# Autoverificacion: misma salida (mismo contenido), reconciliacion sin filtro y lector independiente
for volumen in ("base", "ampliacion"):
    dos = tabla_corridas[tabla_corridas["volumen"] == volumen]
    assert dos["hash"].nunique() == 1, f"las dos corridas de {volumen} deben dejar el mismo hash"
    assert (dos["heredada"] == "OK").all() and (dos["control"] == "OK").all(), f"la capa de {volumen} debe reconciliar con las referencias"
    assert (dos["no_conformes"] == 0).all() and (dos["archivos"] == 120).all(), "0 filas no conformes y un archivo por dia"
    assert independiente[volumen]["solo_en_un_lado"] == 0 and independiente[volumen]["lecturas_distintas"] == 0
    assert independiente[volumen]["diferencia_max_consumo"] <= 1e-6
    assert independiente[volumen]["marca_temporal"]["fisico"] == "INT64"
    assert independiente[volumen]["marca_temporal"]["logico"] == "isAdjustedToUTC=true, unidad=microseconds"
assert (tabla_corridas[tabla_corridas["volumen"] == "base"]["filas"] == 576_000).all()
assert (tabla_corridas[tabla_corridas["volumen"] == "ampliacion"]["filas"] == perfil["lecturas_totales"]).all()
print("Autoverificacion superada: dos corridas por volumen con el mismo hash, reconciliadas sin filtro y con la marca temporal en microsegundos.")

Autoverificacion superada: dos corridas por volumen con el mismo hash, reconciliadas sin filtro y con la marca temporal en microsegundos.


### N2.6 Reproceso de una ventana de 6 días sobre una copia

Con la sobrescritura estática, escribir seis días **borra la tabla completa** y deja solo esos seis; con la dinámica, solo se reemplazan las particiones presentes en lo que se escribe. Las dos corridas escriben en una copia (`--salida`) y anexan a una bitácora aparte (`--bitacora`), porque una corrida parcial no reconcilia con la heredada: es la diferencia que se quiere ver.

In [49]:
# N2.6 Reproceso de 6 dias (2024-01-01 a 2024-01-06) sobre una COPIA de la capa de la base: sobrescritura estatica y dinamica
# Por cada modo: (1) copia la capa completa, (2) ejecuta main con --desde/--hasta/--salida/--sobrescritura, (3) compara los arboles.
# Desde la terminal harian lo mismo con cp -r y dos comandos; ahi deben cerrar antes la sesion del cuaderno (una segunda JVM da OOMKilled).
COPIAS = RAIZ / "_copias_reproceso"
reprocesos = []
for modo in ("estatica", "dinamica"):
    copia = COPIAS / modo
    if copia.exists():
        shutil.rmtree(copia)
    shutil.copytree(LAGO / "refinada_spark", copia)
    arbol_antes = R.resumen_arbol(copia)
    codigo, nuevas = correr_capa("base", desde="2024-01-01", hasta="2024-01-06", salida=copia, sobrescritura=modo, bitacora="bitacora_reproceso.jsonl")
    arbol_despues = R.resumen_arbol(copia)
    intactas, reescritas, perdidas, nuevas_carpetas = R.comparar_arboles(arbol_antes, arbol_despues)
    r = nuevas[-1]
    reprocesos.append({"sobrescritura": modo, "filas_que_quedan": r["filas_escritas"], "carpetas": len(arbol_despues), "intactas": intactas,
                       "reescritas": reescritas, "perdidas": perdidas, "hash": r["hash_salida"][:12], "reconciliacion_heredada": r["reconciliacion_heredada"]})
tabla_reproceso = pd.DataFrame(reprocesos)
print(tabla_reproceso.to_string(index=False))
spark = abrir_sesion()
print("Filas al releer la copia dinamica (el conteo lee solo los pies de los archivos):", spark.read.parquet(str(COPIAS / "dinamica")).count())

sobrescritura  filas_que_quedan  carpetas  intactas  reescritas  perdidas         hash reconciliacion_heredada
     estatica             28800         6         0           6       114 e635456bc71a                   FALLA
     dinamica            576000       120       114           6         0 9cc455c0aff8                      OK
Filas al releer la copia dinamica (el conteo lee solo los pies de los archivos): 576000


In [50]:
# N2.6 (continuacion) El reproceso dinamico con mes y dia como ENTERO: las carpetas mes=1 conviven con mes=01
copia_entero = COPIAS / "mes_entero"
if copia_entero.exists():
    shutil.rmtree(copia_entero)
shutil.copytree(LAGO / "refinada_spark", copia_entero)
spark = abrir_sesion()
ventana = R.transformar(R.leer_cruda(spark, CRUDA, R.definir_esquema()), "nativa")
ventana = ventana.filter((F.col("fecha") >= F.lit("2024-01-01").cast("date")) & (F.col("fecha") <= F.lit("2024-01-06").cast("date")))
R.escribir_capa(ventana.withColumn("mes", F.col("mes").cast("int")).withColumn("dia", F.col("dia").cast("int")), copia_entero, "dinamica")
carpetas_mes = sorted(p.name for p in (copia_entero / "anio=2024").iterdir())
dias_mes_1 = sorted(p.name for p in (copia_entero / "anio=2024" / "mes=1").iterdir())
filas_al_releer = spark.read.parquet(str(copia_entero)).count()
print("Carpetas de mes:", carpetas_mes, "· dias dentro de mes=1:", dias_mes_1)
print(f"Filas al releer: {filas_al_releer} · esperadas: 576000 · duplicadas: {filas_al_releer - 576_000}")

Carpetas de mes: ['mes=01', 'mes=02', 'mes=03', 'mes=04', 'mes=1'] · dias dentro de mes=1: ['dia=1', 'dia=2', 'dia=3', 'dia=4', 'dia=5', 'dia=6']
Filas al releer: 604800 · esperadas: 576000 · duplicadas: 28800


In [51]:
# Autoverificacion del reproceso: la sobrescritura estatica destruye lo que no se reproceso; la dinamica lo conserva
estatica, dinamica = tabla_reproceso.set_index("sobrescritura").loc["estatica"], tabla_reproceso.set_index("sobrescritura").loc["dinamica"]
assert estatica["filas_que_quedan"] == 28_800 and estatica["carpetas"] == 6 and estatica["perdidas"] == 114, "estatica: 6 dias x 4 800 filas"
assert dinamica["filas_que_quedan"] == 576_000 and dinamica["intactas"] == 114 and dinamica["reescritas"] == 6 and dinamica["perdidas"] == 0
assert dinamica["hash"] == tabla_corridas[tabla_corridas["volumen"] == "base"]["hash"].iloc[0], "tras el reproceso dinamico el contenido es el mismo"
assert estatica["reconciliacion_heredada"] == "FALLA" and dinamica["reconciliacion_heredada"] == "OK"
assert "mes=1" in carpetas_mes and "mes=01" in carpetas_mes and filas_al_releer == 604_800, "mes como entero: carpetas mes=1 y mes=01 a la vez y 28 800 filas duplicadas"
print("Autoverificacion superada: 28 800 filas con la estatica, 576 000 y 114 particiones intactas con la dinamica; mes entero duplica 28 800.")

Autoverificacion superada: 28 800 filas con la estatica, 576 000 y 114 particiones intactas con la dinamica; mes entero duplica 28 800.


### N2.7 Comparación de variantes sobre su ampliación

Antes de medir, escriban una hipótesis por tabla: qué variante esperan más rápida y por qué. La celda de las 12 corridas no se ejecuta si las hipótesis siguen siendo `TODO`. Hay dos tablas: **sobre el CSV** (la lectura está incluida: es el costo real del pipeline) y **con la columna de presión en memoria** (queda solo el costo de la función). Si los rangos de dos variantes se solapan, no hay diferencia que declarar.

In [52]:
# N2.7 Escriban su hipotesis ANTES de ejecutar las 12 corridas: que variante sera la mas rapida y por que, en cada tabla
# Escritas antes de medir, con el perfil de equipo-1: 1 002 240 lecturas en 120 CSV (unos 56 MiB de cruda).
HIPOTESIS_VARIANTES = {
    "csv": ("Esperamos que la nativa sea la mas rapida, pero con poca diferencia: cada corrida relee y convierte 120 CSV, "
            "y ese costo (varios segundos) es comun a las tres variantes. La vectorizada deberia quedar con un rango "
            "solapado con la nativa (un lote de Arrow por cada 10 000 filas). La funcion por fila deberia ser la mas lenta, "
            "unas 1,5 a 2 veces la nativa, porque serializa ~1 millon de filas hacia Python, una por una, y su rango no "
            "deberia solaparse con el de la nativa."),
    "memoria": ("Con la columna de presion ya en memoria desaparece el costo de leer el CSV, asi que lo que queda es el "
                "costo de la funcion: esperamos que las diferencias se amplifiquen. La nativa (codigo generado en la JVM) "
                "seria la mas rapida; la vectorizada, unas 2 a 4 veces mas lenta por el viaje a Python en lotes; y la "
                "funcion por fila, 5 a 10 veces la nativa, sin solape entre ninguno de los tres rangos."),
}


In [53]:
# N2.7 (a) Las 12 corridas sobre la CRUDA de su ampliacion, con el script: 1 calentamiento (numero 0) y 3 medidas por variante
assert all(h.strip() and not h.startswith("TODO") for h in HIPOTESIS_VARIANTES.values()), "las hipotesis se escriben ANTES de medir: completen los TODO"
codigo, lineas_variantes = correr_capa("ampliacion", comparar=True)
corridas_variantes = [l for l in lineas_variantes if "numero_corrida" in l]


def medidas_desde_bitacora(registros):
    """Una entrada por variante: calentamiento, medidas, mediana, minimo y maximo (el calentamiento no cuenta)."""
    salida = {}
    for v in R.VARIANTES:
        tiempos = [c["tiempo_s"] for c in sorted((c for c in registros if c["variante"] == v), key=lambda c: c["numero_corrida"])]
        salida[v] = {"calentamiento_s": tiempos[0], "medidas_s": tiempos[1:], "mediana_s": round(statistics.median(tiempos[1:]), 3),
                     "min_s": min(tiempos[1:]), "max_s": max(tiempos[1:])}
    return salida


medidas_equipo_csv = medidas_desde_bitacora(corridas_variantes)
conteos_equipo = {v: [c["conteo_por_clase"] for c in corridas_variantes if c["variante"] == v][-1] for v in R.VARIANTES}
nodos_equipo = {v: corridas_variantes[[c["variante"] for c in corridas_variantes].index(v)]["nodo_plan"] for v in R.VARIANTES}
print("Corridas registradas:", len(corridas_variantes), "· nodos del plan:", nodos_equipo)
print("Conteo por clase:", conteos_equipo["nativa"])
resumen_equipo_csv = tabla_resumen(medidas_equipo_csv)
print(resumen_equipo_csv.to_string())

Corridas registradas: 12 · nodos del plan: {'nativa': 'ninguno', 'udf_fila': 'BatchEvalPython', 'udf_pandas': 'ArrowEvalPython'}
Conteo por clase: {'baja': 47567, 'normal': 906454, 'alta': 48219}
            mediana_s        rango_s  razon_vs_nativa rango_se_solapa_con_nativa
variante                                                                        
nativa          0.614  0.546 a 0.972              1.0                       None
udf_fila        1.478   1.31 a 1.732              2.4                      False
udf_pandas      1.798  1.468 a 2.283              2.9                      False


In [54]:
# N2.7 (b) La misma comparacion con la columna de presion de SU ampliacion en memoria (una celda)
spark = abrir_sesion()
presion_equipo = R.leer_cruda(spark, EQUIPO_CRUDA, R.definir_esquema()).select("presion_bar").cache()
filas_presion_equipo = presion_equipo.count()
medidas_equipo_memoria = {}
for v in R.VARIANTES:
    resultados, resumen = R.medir_variante(spark, presion_equipo, v, 3)
    medidas_equipo_memoria[v] = {"calentamiento_s": resultados[0]["tiempo_s"], "medidas_s": [r["tiempo_s"] for r in resultados[1:]], **resumen}
presion_equipo.unpersist()
resumen_equipo_memoria = tabla_resumen(medidas_equipo_memoria)
print(resumen_equipo_memoria.to_string())

26/11/12 22:06:59 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 1
org.apache.spark.SparkException: EOF reached before Python server acknowledged
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:751)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at org.apache.spark.scheduler.DAGScheduler.updateAccumulators(DAGScheduler.scala:1685)
	at org.apache.spark.scheduler.DAGScheduler.handleTaskCompletion(DAGScheduler.scala:1838)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3054)
	

26/11/12 22:06:59 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 3
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

26/11/12 22:07:00 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 0
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

26/11/12 22:07:00 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 2
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

26/11/12 22:07:01 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 1
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

26/11/12 22:07:02 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 3
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

26/11/12 22:07:02 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 2
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

26/11/12 22:07:02 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 1
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

            mediana_s        rango_s  razon_vs_nativa rango_se_solapa_con_nativa
variante                                                                        
nativa          0.066  0.059 a 0.092              1.0                       None
udf_fila        0.736  0.643 a 0.809             11.2                      False
udf_pandas      0.306  0.279 a 0.313              4.6                      False


26/11/12 22:07:03 ERROR DAGScheduler: Failed to update accumulator 18807 (org.apache.spark.api.python.PythonAccumulatorV2) for task 3
java.net.SocketException: Broken pipe (Write failed)
	at java.base/java.net.SocketOutputStream.socketWrite0(Native Method)
	at java.base/java.net.SocketOutputStream.socketWrite(Unknown Source)
	at java.base/java.net.SocketOutputStream.write(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flushBuffer(Unknown Source)
	at java.base/java.io.BufferedOutputStream.flush(Unknown Source)
	at java.base/java.io.DataOutputStream.flush(Unknown Source)
	at org.apache.spark.api.python.PythonAccumulatorV2.merge(PythonRDD.scala:747)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1(DAGScheduler.scala:1694)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$updateAccumulators$1$adapted(DAGScheduler.scala:1685)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreac

In [55]:
# Autoverificacion: mismo resultado en las tres variantes, protocolo completo y tablas leidas con el criterio del solape
assert len(corridas_variantes) == 12 and sorted({c["numero_corrida"] for c in corridas_variantes}) == [0, 1, 2, 3]
assert conteos_equipo["nativa"] == conteos_equipo["udf_fila"] == conteos_equipo["udf_pandas"], "las tres variantes dan el mismo conteo por clase"
assert sum(conteos_equipo["nativa"].values()) == perfil["lecturas_totales"]
assert nodos_equipo == {"nativa": "ninguno", "udf_fila": "BatchEvalPython", "udf_pandas": "ArrowEvalPython"}
assert filas_presion_equipo == perfil["lecturas_totales"]
assert all(len(m["medidas_s"]) == 3 for m in medidas_equipo_csv.values()) and all(len(m["medidas_s"]) == 3 for m in medidas_equipo_memoria.values())
assert resumen_equipo_memoria.loc["udf_fila", "razon_vs_nativa"] > 1, "con la columna en memoria la funcion por fila cuesta mas que la nativa"
print("Autoverificacion superada: 12 corridas por tabla, mismo conteo por clase y nodos del plan esperados.")

Autoverificacion superada: 12 corridas por tabla, mismo conteo por clase y nodos del plan esperados.


### N2.8 Cierre diario con DataFrame frente a la línea base RDD

Una sola sesión de Spark a la vez: este contraste va **después** de las variantes. El cierre por (fecha, sector) se expresa con `groupBy` de DataFrame sobre `refinada_spark/` de su ampliación y se compara con la corrida de `reduceByKey` con RDD de **su** bitácora de la carga con Spark: trabajos, etapas, tareas y bytes de mezcla. Los DataFrame no se eligen por los bytes de mezcla sino por lo que Spark puede optimizar y por la legibilidad: la cifra decide.

In [56]:
# N2.8 Cierre diario con groupBy de DataFrame sobre refinada_spark/ de SU ampliacion, frente a la linea base RDD de la bitacora de la carga con Spark de su repositorio
spark = abrir_sesion()
capa_equipo = spark.read.parquet(str(EQUIPO_CARPETA / "refinada_spark"))
m = marca()
t0 = time.perf_counter()
cierre = (capa_equipo.groupBy(F.col("fecha").cast("string").alias("fecha"), "sector")
          .agg(F.sum("consumo_m3").alias("consumo_total_m3"), F.count("*").alias("n_lecturas")).collect())
tiempo_cierre = round(time.perf_counter() - t0, 2)
act_df = actividad(m)

bitacora_rdd = pd.read_json(RAIZ / "bitacora_carga_spark.jsonl", lines=True)
candidatas = bitacora_rdd[(bitacora_rdd["volumen"] == "ampliacion") & (bitacora_rdd["metodo"] == "reduceByKey") & (bitacora_rdd["ruta"] == "parquet_refinada")]
linea_base_rdd = candidatas.iloc[-1].to_dict() if len(candidatas) else None
if linea_base_rdd is None:
    print("La bitacora_carga_spark.jsonl que esta usando no tiene una corrida de la ampliacion con reduceByKey sobre Parquet (la del paquete es la de la referencia y solo trae la base): ejecuten  python pipeline/carga_spark.py --equipo", EQUIPO, "--volumen ampliacion  (con la sesion del cuaderno detenida: spark.stop()) y repitan la celda.")
comparacion_cierre = pd.DataFrame([
    {"medicion": "groupBy de DataFrame (hoy)", "trabajos": act_df["trabajos"], "etapas": f"{act_df['etapas']} ({act_df['etapas_omitidas']} omitida)",
     "tareas": act_df["tareas"], "bytes_de_mezcla": act_df["mezcla_escrita_b"], "tiempo_s": tiempo_cierre},
    *([{"medicion": "reduceByKey con RDD (bitacora de la carga con Spark)", "trabajos": "n/d", "etapas": linea_base_rdd["etapas"], "tareas": linea_base_rdd["tareas"],
        "bytes_de_mezcla": linea_base_rdd["mezcla_escritura_bytes"], "tiempo_s": linea_base_rdd["tiempo_s"]}] if linea_base_rdd else [])])
print("Filas del cierre:", len(cierre), "· lecturas por grupo (distintos):", sorted({c["n_lecturas"] for c in cierre})[:3])
print(comparacion_cierre.to_string(index=False))
if linea_base_rdd:
    print("Bytes de mezcla del DataFrame frente al RDD:", round(act_df["mezcla_escrita_b"] / linea_base_rdd["mezcla_escritura_bytes"], 1), "veces")

Filas del cierre: 480 · lecturas por grupo (distintos): [1824, 2880]
                                            medicion trabajos        etapas  tareas  bytes_de_mezcla  tiempo_s
                          groupBy de DataFrame (hoy)        2 2 (1 omitida)       5            41100     0.440
reduceByKey con RDD (bitacora de la carga con Spark)      n/d             2       8             9822     3.282
Bytes de mezcla del DataFrame frente al RDD: 4.2 veces


In [57]:
# Autoverificacion del cierre: el DataFrame gana en tareas y en codigo, pero NO en bytes de mezcla
assert len(cierre) == 480 and {c["n_lecturas"] > 0 for c in cierre} == {True}, "120 dias por 4 sectores"
assert act_df["trabajos"] == 2 and act_df["etapas"] == 2 and act_df["etapas_omitidas"] == 1, "con la ejecucion adaptativa: 2 trabajos, 2 etapas y 1 omitida"
if linea_base_rdd:
    assert act_df["tareas"] < linea_base_rdd["tareas"], "el DataFrame lanza menos tareas que el RDD"
    assert act_df["mezcla_escrita_b"] > linea_base_rdd["mezcla_escritura_bytes"], "el DataFrame mueve MAS bytes de mezcla que el RDD con combinacion previa"
print("Autoverificacion superada: 5 tareas frente a 8, pero mas bytes de mezcla que reduceByKey.")

Autoverificacion superada: 5 tareas frente a 8, pero mas bytes de mezcla que reduceByKey.


### N2.9 Evidencia y registro de decisiones

Primero se escriben las conclusiones (lo **observado**, con la cifra y su origen, separado de lo **interpretado**, la causa). Después una celda genera `evidencia_nivel2_<equipo>.md` con el perfil, las tablas, las hipótesis y las conclusiones. El registro de decisiones usa el formato **ADR** (un texto breve por decisión, con contexto, opciones, evidencia, decisión, alternativa descartada con su costo, consecuencias y la cifra que obligaría a revisarla) y justifica con cifras de **su ampliación** al menos cuatro de las cinco decisiones elegibles, entre ellas el esquema (1) y la función propia frente a la nativa (4).

**Antes de la celda de evidencia.** Esa celda usa variables de celdas anteriores (`tabla_corridas`, `tabla_reproceso`, `resumen_equipo_csv`, `comparacion_cierre`...). Si reinician el núcleo, esas variables se pierden: vuelvan a ejecutar las celdas que las producen, o guarden sus tablas antes de reiniciar.

In [58]:
# N2.9 La bitacora: una linea por corrida. Las corridas de la capa llevan reconciliacion; las de variantes, numero de corrida y nodo del plan
bitacora = leer_bitacora()
capas = pd.DataFrame([b for b in bitacora if "reconciliacion_heredada" in b])
print(capas[["equipo", "volumen", "variante", "particiones_entrada", "trabajos", "etapas", "tareas", "mib_leidos", "filas_escritas", "archivos_escritos", "reconciliacion_heredada", "reconciliacion_control"]].to_string(index=False))
print("\nCorridas de variantes registradas:", sum(1 for b in bitacora if "numero_corrida" in b), "· campos de una corrida de la capa:")
print(sorted(next(b for b in bitacora if "reconciliacion_heredada" in b)))

  equipo    volumen variante  particiones_entrada  trabajos  etapas  tareas  mib_leidos  filas_escritas  archivos_escritos reconciliacion_heredada reconciliacion_control
equipo-1       base   nativa                    4         4       4      13       56.48          576000                120                      OK                     OK
equipo-1       base   nativa                    4         4       4      13       56.48          576000                120                      OK                     OK
equipo-1 ampliacion   nativa                    5         4       4      15      103.71         1002240                120                      OK                     OK
equipo-1 ampliacion   nativa                    5         4       4      15      103.71         1002240                120                      OK                     OK

Corridas de variantes registradas: 12 · campos de una corrida de la capa:
['archivos_escritos', 'detalle_consolidada_spark', 'detalle_control', 'deta

In [59]:
# N2.9 Conclusiones del equipo: separen lo OBSERVADO (cifras de las tablas) de lo INTERPRETADO (la causa)
CONCLUSIONES = {
    "csv": "OBSERVADO: sobre el CSV de la ampliacion (1 002 240 filas, 5 particiones), las medianas fueron: nativa 0,61 s (rango 0,55 a 0,97), por fila 1,48 s (1,31 a 1,73; 2,4 veces) y vectorizada 1,80 s (1,47 a 2,28; 2,9 veces). Ninguno de los dos rangos de las funciones propias se solapa con el de la nativa; los de las dos funciones propias SI se solapan entre si. El conteo por clase es identico en las tres (47 567 / 906 454 / 48 219). En la corrida anterior del mismo cuaderno el orden de las dos funciones propias habia salido al reves (vectorizada 0,84 s y por fila 0,94 s). INTERPRETADO: se cumple la parte firme de la hipotesis (la nativa es la mas rapida, sin solape) y la funcion por fila queda en 2,4 veces, algo por encima del 1,5 a 2 estimado. Se refuta que la vectorizada se solapara con la nativa. Entre por fila y vectorizada NO hay diferencia que declarar con el CSV: sus rangos se solapan y su orden cambio entre dos ejecuciones del cuaderno, porque la lectura y la conversion del CSV (comun a las tres) domina y mete ruido.",
    "memoria": "OBSERVADO: con la columna de presion en memoria, las medianas fueron: nativa 0,066 s, vectorizada 0,306 s (4,6 veces) y por fila 0,736 s (11,2 veces), sin solape entre ningun par de rangos; el orden coincide con el de la corrida anterior (5,5 y 9,8 veces). INTERPRETADO: se cumple la hipotesis del orden, y aqui la diferencia si es estable porque sin la lectura del CSV queda solo el costo de la funcion. La nativa se evalua en la JVM (sin nodo de Python en el plan); la vectorizada paga 104 lotes de Arrow (ArrowEvalPython); la funcion por fila serializa y llama 1 002 240 veces a Python (BatchEvalPython). Las razones observadas (4,6 y 11,2) quedaron por encima de lo estimado (2 a 4 y 5 a 10).",
    "reproceso": "OBSERVADO: al reprocesar 6 dias (2024-01-01 a 2024-01-06) sobre una copia de la capa de la base, la sobrescritura estatica dejo 28 800 filas (6 x 4 800) y 6 carpetas: perdio 114 de 120 particiones y la reconciliacion con la heredada FALLA. La dinamica dejo 576 000 filas, 120 carpetas (114 intactas, 6 reescritas, 0 perdidas) y el mismo hash 9cc455c0aff8 que la capa original. Con mes y dia como entero aparecen mes=1 junto a mes=01 y al releer hay 604 800 filas (28 800 duplicadas). INTERPRETADO: la estatica interpreta 'overwrite' como 'reemplazar toda la tabla'; la dinamica, como 'reemplazar solo las particiones que traigo'. Adoptamos la dinamica con rutas de texto con ceros.",
    "cierre": "OBSERVADO: el cierre diario con groupBy de DataFrame sobre refinada_spark/ de la ampliacion lanzo 5 tareas en 2 etapas (1 omitida) y movio 41 100 B de mezcla en 0,44 s. El RDD con reduceByKey de nuestra T15 lanzo 8 tareas y movio 9 822 B en 3,28 s (ese tiempo incluye leer y convertir el Parquet heredado). El DataFrame mueve 4,2 veces mas bytes. INTERPRETADO: el DataFrame gana en tareas (AQE fusiona las particiones de mezcla), en tiempo y en codigo (una linea, sin lambdas), pero no en bytes: su agregacion parcial serializa filas de Tungsten con claves y acumuladores tipados, y el RDD solo serializa tuplas pequenas. Con 480 grupos, la diferencia de 31 KB es irrelevante: elegiriamos el DataFrame por mantenibilidad y por el optimizador.",
}


In [60]:
# N2.9 Evidencia del Nivel 2: se genera el archivo evidencia_nivel2_<equipo>.md con lo medido y lo concluido
# Usa variables de celdas anteriores (tabla_corridas, tabla_reproceso, resumen_equipo_csv, comparacion_cierre...).
# Si reiniciaron el nucleo, esas variables ya no existen: vuelvan a ejecutar las celdas que las producen antes de esta.
assert set(CONCLUSIONES) == {"csv", "memoria", "reproceso", "cierre"}, "escriban las cuatro conclusiones"
assert all(c.strip() and not c.startswith("TODO") for c in CONCLUSIONES.values()), "completen las conclusiones: separen lo observado de lo interpretado"


def md_tabla(tabla):
    """Tabla en markdown, sin dependencias adicionales."""
    tabla = tabla.reset_index() if tabla.index.name else tabla
    filas = ["| " + " | ".join(str(v) for v in fila) + " |" for fila in tabla.itertuples(index=False)]
    return "\n".join(["| " + " | ".join(map(str, tabla.columns)) + " |", "|" + "---|" * len(tabla.columns), *filas])


partes = [f"# Evidencia del Nivel 2 · {EQUIPO}", "",
          "## Perfil de la ampliacion", "",
          f"- Comando: `python practica/generar_ampliacion_s15.py --equipo {EQUIPO}` ({tiempo_generacion} s)",
          f"- Medidores: {perfil['medidores']} · lecturas (total, no incremento): {perfil['lecturas_totales']}",
          f"- Sector dominante: {perfil['sector_dominante']} · fraccion de medidores: {perfil['fraccion_dominante']} · disco: {disco_mib} MiB",
          f"- Reparto de la cruda: {len(filas_equipo)} particiones, archivos por particion {archivos_equipo}, filas por particion {filas_equipo}, lotes de Arrow esperados {lotes_equipo}", "",
          "## Capa reejecutable y reconciliacion (dos corridas por volumen)", "", md_tabla(tabla_corridas), "",
          "## Lector independiente (pyarrow y pandas, sin Spark)", "", md_tabla(tabla_independiente.drop(columns="marca_temporal").reset_index(names="volumen")), "",
          f"Marca temporal de la ampliacion: {independiente['ampliacion']['marca_temporal']}", "",
          "## Reproceso de 6 dias sobre una copia (base)", "", md_tabla(tabla_reproceso), "",
          f"Con mes y dia como entero: carpetas de mes {carpetas_mes}, {filas_al_releer} filas al releer ({filas_al_releer - 576_000} duplicadas).", "",
          f"- **Conclusion:** {CONCLUSIONES['reproceso']}", "",
          "## Variantes de clasificacion", "",
          f"- **Hipotesis sobre el CSV (escrita antes de medir):** {HIPOTESIS_VARIANTES['csv']}",
          f"- **Hipotesis con la columna en memoria:** {HIPOTESIS_VARIANTES['memoria']}", "",
          "### Sobre la cruda en CSV (ampliacion, 12 corridas: 1 calentamiento descartado y 3 medidas por variante)", "", md_tabla(resumen_equipo_csv), "",
          f"- **Conclusion:** {CONCLUSIONES['csv']}", "",
          "### Con la columna de presion en memoria", "", md_tabla(resumen_equipo_memoria), "",
          f"- **Conclusion:** {CONCLUSIONES['memoria']}", "",
          "## Cierre diario: DataFrame frente a la linea base RDD", "", md_tabla(comparacion_cierre), "",
          f"- **Conclusion:** {CONCLUSIONES['cierre']}", ""]
ruta_evidencia = RAIZ / f"evidencia_nivel2_{EQUIPO}.md"
ruta_evidencia.write_text("\n".join(partes), encoding="utf-8")
print("Escrito:", ruta_evidencia, f"({ruta_evidencia.stat().st_size} bytes)")

Escrito: /proyecto/evidencia_nivel2_equipo-1.md (6689 bytes)


In [61]:
# Plantilla del registro de decisiones (ADR: un texto breve por decision). Copien este texto a su repositorio, una vez por decision, y completenlo con SUS cifras.
PLANTILLA_ADR = """# ADR-N: <titulo corto de la decision>
Estado: propuesta | aceptada
Contexto: <que se decide, con que volumen (cifras de su ampliacion) y que pregunta de la gerencia lo detono>
Opciones: <los valores o alternativas que midieron>
Evidencia (cifra y de donde sale: bitacora, tabla, salida de pyarrow): <trabajos, MiB leidos, nulos, tipo fisico y logico, mediana y rango, filas que sobreviven...>
Decision: <que eligieron y en que condiciones vale>
Alternativa descartada y su costo: <que no eligieron y que costaba, con cifra medida o calculada>
Consecuencias: <que se gana y que se arriesga con esta decision>
Cifra que obligaria a revisarla: <por ejemplo, un volumen, una proporcion de nulos o un cambio en el encabezado de la fuente>"""
print(PLANTILLA_ADR)

# ADR-N: <titulo corto de la decision>
Estado: propuesta | aceptada
Contexto: <que se decide, con que volumen (cifras de su ampliacion) y que pregunta de la gerencia lo detono>
Opciones: <los valores o alternativas que midieron>
Evidencia (cifra y de donde sale: bitacora, tabla, salida de pyarrow): <trabajos, MiB leidos, nulos, tipo fisico y logico, mediana y rango, filas que sobreviven...>
Decision: <que eligieron y en que condiciones vale>
Alternativa descartada y su costo: <que no eligieron y que costaba, con cifra medida o calculada>
Consecuencias: <que se gana y que se arriesga con esta decision>
Cifra que obligaria a revisarla: <por ejemplo, un volumen, una proporcion de nulos o un cambio en el encabezado de la fuente>


### N2.10 Verificación por etapa y qué se versiona

`pipeline/verificar_etapa.py` comprueba cada etapa del pipeline y dice el valor observado frente al esperado. Las etapas t15 y t16 corren aquí; t12 a t14 necesitan el perfil `s12` (ver la guía).

In [62]:
# N2.10 Verificacion por etapa: el comando unico que pegan en su indice de evidencias (t15 y t16 corren aqui; t12 a t14 necesitan el perfil s12)
from pipeline import verificar_etapa as V

salidas_verificacion = {}
for etapa in ("t15", "t16"):
    texto = io.StringIO()
    with contextlib.redirect_stdout(texto):
        codigo = V.main(["--etapa", etapa, "--raiz", str(RAIZ)])
    salidas_verificacion[etapa] = (codigo, texto.getvalue())
    print(texto.getvalue().strip().splitlines()[-1])

Etapa t15: APROBADA (6 de 6 comprobaciones, 0.1 s)


Etapa t16: APROBADA (15 de 15 comprobaciones, 0.9 s)


In [63]:
# Autoverificacion: las dos etapas que corren sin PostgreSQL deben quedar aprobadas
assert salidas_verificacion["t15"][0] == 0, salidas_verificacion["t15"][1]
assert salidas_verificacion["t16"][0] == 0, salidas_verificacion["t16"][1]
print("Etapas t15 y t16 aprobadas: peguen estas lineas en su indice de evidencias.")

Etapas t15 y t16 aprobadas: peguen estas lineas en su indice de evidencias.


In [64]:
# N2.11 Que se versiona: el repositorio lleva texto y resultados pequenos, no artefactos que otra persona puede regenerar
candidatos = ["pipeline/refinada_spark.py", "pipeline/verificar_etapa.py", "bitacora_refinada_spark.jsonl", "bitacora_reproceso.jsonl", "mediciones_nivel1.csv",
              f"evidencia_nivel2_{EQUIPO}.md", "datalake/refinada_spark", "datalake_equipo/refinada_spark", "datalake_equipo/totales_control.csv"]
excluidos = ["datalake/cruda", "datalake/refinada", "datalake_equipo/cruda", "datalake_equipo/refinada", "_copias_reproceso", ".ipynb_checkpoints"]


def tamano(ruta):
    ruta = RAIZ / ruta
    if ruta.is_dir():
        return sum(f.stat().st_size for f in ruta.rglob("*") if f.is_file())
    return ruta.stat().st_size if ruta.exists() else None


print("SE VERSIONA")
for c in candidatos:
    t = tamano(c)
    print(f"  {c:<45} {'no existe' if t is None else f'{t / 1024:,.0f} KiB'}")
print("NO SE VERSIONA (se regenera)")
for c in excluidos:
    t = tamano(c)
    print(f"  {c:<45} {'no existe' if t is None else f'{t / 2**20:,.1f} MiB'}")

SE VERSIONA
  pipeline/refinada_spark.py                    29 KiB
  pipeline/verificar_etapa.py                   13 KiB
  bitacora_refinada_spark.jsonl                 9 KiB
  bitacora_reproceso.jsonl                      2 KiB
  mediciones_nivel1.csv                         1 KiB
  evidencia_nivel2_equipo-1.md                  7 KiB
  datalake/refinada_spark                       2,960 KiB
  datalake_equipo/refinada_spark                4,449 KiB
  datalake_equipo/totales_control.csv           15 KiB
NO SE VERSIONA (se regenera)
  datalake/cruda                                28.2 MiB
  datalake/refinada                             2.8 MiB
  datalake_equipo/cruda                         51.9 MiB
  datalake_equipo/refinada                      4.2 MiB


  _copias_reproceso                             6.1 MiB
  .ipynb_checkpoints                            no existe


## Nivel 3 · Frontera · opcional

**La decisión (sin guía).** ¿Qué debe hacer el pipeline si la fuente cambia el CSV: agrega una columna, quita una, reordena dos, o cambia el formato de la fecha? Las opciones son **fallar**, **absorber** el cambio (por ejemplo, con `mergeSchema` en una lectura de Parquet) o **mandar el archivo a cuarentena**. Las celdas siguientes crean copias modificadas de un solo día de la cruda y miden qué hace la lectura declarada con cada una; ustedes escriben la política en su registro de decisiones (como una sexta decisión, adicional: no reemplaza a ninguna de las cuatro obligatorias de la tarea) y una función que detecta el cambio en el encabezado antes de leer.

In [65]:
# Nivel 3 · Preparacion: copias modificadas de UN dia de la cruda (el dia 2024-01-05) con los cambios que una fuente puede hacer
N3 = Path(tempfile.mkdtemp(prefix="n3_"))
origen_dia = CRUDA / "anio=2024" / "mes=01" / "dia=05" / "lecturas.csv"
original = pd.read_csv(origen_dia)
casos = {
    "sin_cambio": original,
    "columna_nueva": original.assign(zona="Z1"),                                            # una columna al final
    "columna_menos": original.drop(columns=["quality_flag"]),                                # una columna eliminada
    "orden_distinto": original[["meter_id", "timestamp", "sector", "presion_bar", "consumo_m3", "quality_flag"]],
    "fecha_otro_formato": original.assign(timestamp=pd.to_datetime(original["timestamp"]).dt.strftime("%d/%m/%Y %H:%M:%S")),   # el tipo de la marca cambia
}
rutas_n3 = {}
for nombre, tabla in casos.items():
    carpeta = N3 / nombre / "anio=2024" / "mes=01" / "dia=05"
    carpeta.mkdir(parents=True)
    tabla.to_csv(carpeta / "lecturas.csv", index=False, lineterminator="\r\n")
    rutas_n3[nombre] = N3 / nombre
print("Copias creadas:", list(rutas_n3))

Copias creadas: ['sin_cambio', 'columna_nueva', 'columna_menos', 'orden_distinto', 'fecha_otro_formato']


In [66]:
# Nivel 3 · Que hace la lectura declarada con cada cambio: con enforceSchema falso (como en el script) y con verdadero (la omision)
spark = abrir_sesion()
resultados_n3 = []
for nombre, ruta in rutas_n3.items():
    for enforce in (False, True):
        df = spark.read.csv(str(ruta), header=True, schema=ESQUEMA, mode="PERMISSIVE", enforceSchema=enforce, timestampFormat="yyyy-MM-dd HH:mm:ss")
        resultados_n3.append({"caso": nombre, "enforceSchema": enforce, "count()": intentar(df.count),
                              "nulos por columna": intentar(lambda df=df: {c: n for c, n in R.resumen_calidad(df)["nulos"].items() if n}),
                              "consumo_m3 total": intentar(lambda df=df: round(df.agg(F.sum("consumo_m3")).collect()[0][0], 1) if df.agg(F.sum("consumo_m3")).collect()[0][0] is not None else None)})
tabla_n3 = pd.DataFrame(resultados_n3)
print(tabla_n3.to_string(index=False))

26/11/12 22:07:06 ERROR Executor: Exception in task 0.0 in stage 71.0 (TID 107)
java.lang.IllegalArgumentException: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 7, schema size: 6
CSV file: file:///tmp/n3_pxjui_1o/columna_nueva/anio=2024/mes=01/dia=05/lecturas.csv
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2(CSVHeaderChecker.scala:95)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2$adapted(CSVHeaderChecker.scala:91)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.checkHeaderColumnNames(CSVHeaderChecker.scala:91)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$6(CSVHeaderChecker.scala:127)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$6$adapted(CSVHeaderChecker.scala:126)
	at scala.Option.foreach(Option.scala:407)
	at org.apach

26/11/12 22:07:06 ERROR Executor: Exception in task 0.0 in stage 93.0 (TID 122)
java.lang.IllegalArgumentException: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 5, schema size: 6
CSV file: file:///tmp/n3_pxjui_1o/columna_menos/anio=2024/mes=01/dia=05/lecturas.csv
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2(CSVHeaderChecker.scala:95)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2$adapted(CSVHeaderChecker.scala:91)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.checkHeaderColumnNames(CSVHeaderChecker.scala:91)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$6(CSVHeaderChecker.scala:127)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$6$adapted(CSVHeaderChecker.scala:126)
	at scala.Option.foreach(Option.scala:407)
	at org.apach

26/11/12 22:07:07 ERROR Executor: Exception in task 0.0 in stage 115.0 (TID 137)
java.lang.IllegalArgumentException: CSV header does not conform to the schema.
 Header: meter_id, timestamp, sector, presion_bar, consumo_m3, quality_flag
 Schema: meter_id, timestamp, sector, consumo_m3, presion_bar, quality_flag
Expected: consumo_m3 but found: presion_bar
CSV file: file:///tmp/n3_pxjui_1o/orden_distinto/anio=2024/mes=01/dia=05/lecturas.csv
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2(CSVHeaderChecker.scala:95)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$2$adapted(CSVHeaderChecker.scala:91)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.checkHeaderColumnNames(CSVHeaderChecker.scala:91)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecker.$anonfun$checkHeaderColumnNames$6(CSVHeaderChecker.scala:127)
	at org.apache.spark.sql.catalyst.csv.CSVHeaderChecke

              caso  enforceSchema  count()                  nulos por columna                   consumo_m3 total
        sin_cambio          False     4800                                 {}                              768.6
        sin_cambio           True     4800                                 {}                              768.6
     columna_nueva          False     4800    ERROR: IllegalArgumentException                              768.6
     columna_nueva           True     4800                                 {}                              768.6
     columna_menos          False     4800    ERROR: IllegalArgumentException                              768.6
     columna_menos           True     4800             {'quality_flag': 4800}                              768.6
    orden_distinto          False     4800 ERROR: CSV header does not conform ERROR: CSV header does not conform
    orden_distinto           True     4800                                 {}                   

In [67]:
def clasificar_encabezado(encabezado, esquema):
    """Compara el encabezado de un CSV (lista de nombres) con el esquema declarado y devuelve uno de:
    'sin_cambio', 'columna_nueva', 'columna_menos' u 'orden_distinto'.

    TODO (equipo): implementen la politica de deteccion que van a escribir en su registro de decisiones.
    Un encabezado con las mismas columnas en otro orden es 'orden_distinto'; con una columna que el esquema
    no tiene es 'columna_nueva'; con una columna menos es 'columna_menos'.
    """
    nombres = [f.name for f in esquema.fields]
    esperado, recibido = set(nombres), set(encabezado)
    if recibido - esperado:
        return "columna_nueva"
    if esperado - recibido:
        return "columna_menos"
    if list(encabezado) != nombres:
        return "orden_distinto"
    return "sin_cambio"

In [68]:
# Autoverificacion del Nivel 3: la clasificacion detecta en el ENCABEZADO lo que la lectura declarada deja pasar o rompe
for nombre, ruta in rutas_n3.items():
    encabezado = pd.read_csv(ruta / "anio=2024" / "mes=01" / "dia=05" / "lecturas.csv", nrows=0).columns.tolist()
    esperado = {"sin_cambio": "sin_cambio", "columna_nueva": "columna_nueva", "columna_menos": "columna_menos",
                "orden_distinto": "orden_distinto", "fecha_otro_formato": "sin_cambio"}[nombre]
    assert clasificar_encabezado(encabezado, ESQUEMA) == esperado, f"{nombre}: se esperaba {esperado}"
print("Autoverificacion superada. Recuerden: el cambio de formato de la fecha NO se ve en el encabezado; se ve en los nulos por columna.")
shutil.rmtree(N3)

Autoverificacion superada. Recuerden: el cambio de formato de la fecha NO se ve en el encabezado; se ve en los nulos por columna.


## Reto de diseño · 30 minutos · opcional · Power Humanise

**Situación.** La dirección pregunta qué pasaría con la función por fila si el acueducto pasara de 200 a 100 000 medidores. La respuesta tiene que caber en 3 minutos y en palabras que entienda un comité: la celda calcula las invocaciones y extrapola un tiempo con un supuesto declarado; ustedes preparan el guion y una **tercera pregunta difícil** que la nota de una página no incluye.

In [69]:
def invocaciones_por_cierre(medidores, lecturas_por_medidor_y_dia=24, variante="udf_fila", particiones=4, lote=10_000):
    """Invocaciones de la funcion propia en un cierre diario.

    TODO (equipo): variante 'udf_fila' = una invocacion por fila; 'udf_pandas' = una por lote de Arrow, con
    lotes = particiones * techo(filas por particion / lote); 'nativa' = 0.
    """
    import math
    filas = medidores * lecturas_por_medidor_y_dia
    if variante == "udf_fila":
        return filas
    if variante == "udf_pandas":
        return particiones * math.ceil(filas / particiones / lote)
    return 0


def extrapolar_segundos(mediana_s, filas_medidas, filas_objetivo):
    """Tiempo estimado suponiendo que crece de forma lineal con las filas (SUPUESTO que deben declarar)."""
    return round(mediana_s * filas_objetivo / filas_medidas, 1)

In [70]:
# Autoverificacion del reto: la base (un dia y los 120 dias) y un cierre de 100 000 medidores
assert invocaciones_por_cierre(200, variante="udf_fila") == 4_800 and invocaciones_por_cierre(200, variante="nativa") == 0
assert invocaciones_por_cierre(200, variante="udf_pandas") == 4, "un dia de la base: 4 800 filas en 4 particiones de 1 200 dan 4 lotes"
assert invocaciones_por_cierre(200, lecturas_por_medidor_y_dia=24 * 120, variante="udf_pandas") == 60, "los 120 dias de la base: 576 000 filas en 4 particiones de 144 000 dan 60 lotes"
assert invocaciones_por_cierre(100_000, variante="udf_fila") == 2_400_000
assert invocaciones_por_cierre(100_000, variante="udf_pandas") == 240
assert extrapolar_segundos(1.0, 576_000, 2_400_000) == 4.2
print("Autoverificacion superada: 2 400 000 invocaciones por cierre con la funcion por fila frente a 240 lotes de Arrow.")

Autoverificacion superada: 2 400 000 invocaciones por cierre con la funcion por fila frente a 240 lotes de Arrow.


## Antes de entregar

Suban al repositorio del equipo: `pipeline/refinada_spark.py` completo, `datalake/refinada_spark/`, `datalake_equipo/refinada_spark/`, `bitacora_refinada_spark.jsonl`, `mediciones_nivel1.csv`, `evidencia_nivel2_<equipo>.md`, el registro de decisiones, el contraste del sub-paso A y el índice de evidencias con su README. La celda siguiente comprueba lo que se puede comprobar con código.

In [71]:
# Lista de entrega: lo que debe existir en el repositorio del equipo antes de entregar
lista = {
    "pipeline/refinada_spark.py completo (sin NotImplementedError)": "NotImplementedError" not in (RAIZ / "pipeline" / "refinada_spark.py").read_text(encoding="utf-8"),
    "datalake/refinada_spark/ con 120 carpetas de dia": len(list((LAGO / "refinada_spark").glob("anio=*/mes=*/dia=*"))) == 120,
    "datalake_equipo/refinada_spark/ con 120 carpetas de dia": len(list((RAIZ / "datalake_equipo" / "refinada_spark").glob("anio=*/mes=*/dia=*"))) == 120,
    "bitacora_refinada_spark.jsonl con dos corridas por volumen y las de variantes": len([b for b in leer_bitacora() if "reconciliacion_heredada" in b]) >= 4 and len([b for b in leer_bitacora() if "numero_corrida" in b]) >= 12,
    "mediciones_nivel1.csv": (RAIZ / "mediciones_nivel1.csv").exists(),
    f"evidencia_nivel2_{EQUIPO}.md": (RAIZ / f"evidencia_nivel2_{EQUIPO}.md").exists(),
    "datalake/ heredado identico": hash_arbol(LAGO) == (datalake_antes, archivos_antes),
}
for item, ok in lista.items():
    print("[x]" if ok else "[ ]", item)
print("\nFalta, fuera del cuaderno: registro de decisiones, nota para la gerencia, puente de modelado, indice de evidencias, README y lectura en ingles.")

[x] pipeline/refinada_spark.py completo (sin NotImplementedError)
[x] datalake/refinada_spark/ con 120 carpetas de dia
[x] datalake_equipo/refinada_spark/ con 120 carpetas de dia
[x] bitacora_refinada_spark.jsonl con dos corridas por volumen y las de variantes
[x] mediciones_nivel1.csv
[x] evidencia_nivel2_equipo-1.md
[x] datalake/ heredado identico

Falta, fuera del cuaderno: registro de decisiones, nota para la gerencia, puente de modelado, indice de evidencias, README y lectura en ingles.


In [72]:
# Al terminar: cierren la sesion de Spark y apaguen el servicio desde la terminal del computador
spark.stop()
print("Sesion cerrada. En la terminal del computador:  docker compose --profile s16 down   (los archivos del paquete se conservan)")

Sesion cerrada. En la terminal del computador:  docker compose --profile s16 down   (los archivos del paquete se conservan)
